# Overview

This notebook implements a full ML pipeline:
- data processing
- feature engineering
- modeling
- evaluation

The focus is on maintaining data integrity and exploring model performance across multiple configurations.

# 1. Setup & Imports

This section contains all required libraries and initial setup.

In [1]:
import pandas as pd
import gc
import os
import json
import dask.dataframe as dd
from joblib import Parallel, delayed
import multiprocessing
import joblib

from tqdm import tqdm

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split, GridSearchCV
from itertools import product
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

import numpy as np
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from catboost import Pool
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import lightgbm as lgb
from xgboost import XGBClassifier, callback
import xgboost as xgb

In [2]:
path = r"C:\Users\User\-_PRACTICAL\junior-final-work\train_data"

In [3]:
sample_120k = pd.read_csv('sample_120k.csv')

In [3]:
data = pd.read_csv('full_data13.csv')

# 2. Data Pipeline Functions

This section defines reusable functions for:
- loading and merging data from multiple files
- memory optimization
- feature engineering pipeline

Note: functions are grouped together for convenience, although they cover multiple pipeline stages.

## functions

In [8]:
def reduce_mem_usage(df, int_cast=True, obj_to_category=True, subset=None):
    """
    Iterate through all the columns of a dataframe and modify the data type to reduce memory usage.
    :param df: dataframe to reduce (pd.DataFrame)
    :param int_cast: indicate if columns should be tried to be casted to int (bool)
    :param obj_to_category: convert non-datetime related objects to category dtype (bool)
    :param subset: subset of columns to analyse (list)
    :return: dataset with the column dtypes adjusted (pd.DataFrame)
    """
    start_mem = df.memory_usage().sum() / 1024 ** 2;
    gc.collect()
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))

    cols = subset if subset is not None else df.columns.tolist()

    for col in tqdm(cols):

        col_type = df[col].dtype

        if col_type != object and col_type.name != 'category' and 'datetime' not in col_type.name:
            c_min = df[col].min()
            c_max = df[col].max()

            # test if column can be converted to an integer
            treat_as_int = str(col_type)[:3] == 'int'
            # if int_cast and not treat_as_int:
            #     treat_as_int = check_if_integer(df[col])

            if treat_as_int:
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.uint8).min and c_max < np.iinfo(np.uint8).max:
                    df[col] = df[col].astype(np.uint8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.uint16).min and c_max < np.iinfo(np.uint16).max:
                    df[col] = df[col].astype(np.uint16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.uint32).min and c_max < np.iinfo(np.uint32).max:
                    df[col] = df[col].astype(np.uint32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
                elif c_min > np.iinfo(np.uint64).min and c_max < np.iinfo(np.uint64).max:
                    df[col] = df[col].astype(np.uint64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        elif 'datetime' not in col_type.name and obj_to_category:
            df[col] = df[col].astype('category')
    gc.collect()
    end_mem = df.memory_usage().sum() / 1024 ** 2
    print('Memory usage after optimization is: {:.3f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))

    return df

In [9]:
def read_parquet_dataset_from_local(path_to_dataset: str, file_name_tag: str, start_from: int = 0,
                                     num_parts_to_read: int = 2, columns=None, verbose=False) -> pd.DataFrame:
    """
    читает num_parts_to_read партиций, преобразует их к pd.DataFrame и возвращает
    :param path_to_dataset: путь до директории с партициями
    :param start_from: номер партиции, с которой начать чтение
    :param num_parts_to_read: количество партиций, которые требуется прочитать
    :param columns: список колонок, которые нужно прочитать из партиции
    :return: pd.DataFrame
    """

    res = []
    dataset_paths = sorted([os.path.join(path_to_dataset, filename) for filename in os.listdir(path_to_dataset)
                              if filename.startswith(file_name_tag)])

    start_from = max(0, start_from)
    chunks = dataset_paths[start_from: start_from + num_parts_to_read]
    if verbose:
        print('Reading chunks:\n')
        for chunk in chunks:
            print(chunk)
    for chunk_path in tqdm(chunks, desc="Reading dataset with pandas"):
        print('chunk_path', chunk_path)
        chunk = pd.read_parquet(chunk_path,columns=columns)
        chunk = reduce_mem_usage(chunk)
        res.append(chunk)
        del chunk
        gc.collect()

    return pd.concat(res).fillna(0).reset_index(drop=True)

### pipeline

In [6]:
def prepare_transactions_dataset(path_to_dataset: str, file_name_tag: str, num_parts_to_preprocess_at_once: int = 1, num_parts_total: int=50,
                                 save_to_path=None, verbose: bool=False):
    """
    возвращает готовый pd.DataFrame с признаками, на которых можно учить модель для целевой задачи.
    path_to_dataset: str
        путь до датасета с партициями
    num_parts_to_preprocess_at_once: int
        количество партиций, которые будут одновременно держаться в памяти и обрабатываться
    num_parts_total: int
        общее количество партиций, которые нужно обработать
    save_to_path: str
        путь до папки, в которой будет сохранен каждый обработанный блок в .parquet формате. Если None, то не будет сохранен
    verbose: bool
        логирует каждый обрабатываемый кусок данных
    """
    preprocessed_frames = []

    for step in tqdm(range(0, num_parts_total, num_parts_to_preprocess_at_once),
                               desc="Transforming transactions data"):
        transactions_frame = read_parquet_dataset_from_local(
            path_to_dataset, file_name_tag, step, num_parts_to_preprocess_at_once,
            verbose=verbose
        )

        # === базовые колонки ===
        paym_cols = [f"enc_paym_{i}" for i in range(24)]  # FIX: используем все 0..23

        # === платежные фичи ===
        transactions_frame["paym_bad_count"] = (transactions_frame[paym_cols] > 0).sum(axis=1)
        transactions_frame["paym_severe_count"] = (transactions_frame[paym_cols] >= 2).sum(axis=1)
        transactions_frame["paym_nonzero_share"] = (transactions_frame[paym_cols] > 0).mean(axis=1)
        transactions_frame["paym_mean_status"] = transactions_frame[paym_cols].mean(axis=1)
        transactions_frame["enc_paym_weighted_score"] = transactions_frame[paym_cols].sum(axis=1)

        # === сортировка (ВАЖНО для last) ===
        transactions_frame = transactions_frame.sort_values(['id', 'rn'])

        # === клиентские агрегаты (до groupby) ===
        grp = transactions_frame.groupby('id')

        # utilization
        transactions_frame['utilization_ratio'] = (
            transactions_frame['pre_loans_outstanding'] /
            (transactions_frame['pre_loans_credit_limit'] + 1e-6)
        )

        # last loan flag
        transactions_frame["is_last_loan"] = (
            transactions_frame["rn"] == grp["rn"].transform("max")
        ).astype(int)

        # previous loan features
        transactions_frame["prev_util"] = grp["utilization_ratio"].shift(1)
        transactions_frame["prev_paym"] = grp["paym_mean_status"].shift(1)

        transactions_frame["last_vs_prev_util"] = (
            transactions_frame["utilization_ratio"] - transactions_frame["prev_util"]
        )
        transactions_frame["last_vs_prev_paym"] = (
            transactions_frame["paym_mean_status"] - transactions_frame["prev_paym"]
        )

        # ranks (позиция кредита внутри клиента)
        transactions_frame["util_rank"] = grp["utilization_ratio"].rank(pct=True)
        transactions_frame["paym_rank"] = grp["paym_mean_status"].rank(pct=True)

        # rolling last 3 loans (локальный контекст)
        transactions_frame["util_last3_mean"] = grp["utilization_ratio"].transform(
            lambda x: x.rolling(3, min_periods=1).mean()
        )

        transactions_frame["paym_last3_mean"] = grp["paym_mean_status"].transform(
            lambda x: x.rolling(3, min_periods=1).mean()
        )

        transactions_frame["last_vs_last3_util"] = (
            transactions_frame["utilization_ratio"] - transactions_frame["util_last3_mean"]
        )

        transactions_frame["last_vs_last3_paym"] = (
            transactions_frame["paym_mean_status"] - transactions_frame["paym_last3_mean"]
        )

        transactions_frame['client_products_count'] = grp['id'].transform('count')

        transactions_frame['mean_utilization_ratio'] = grp['utilization_ratio'].transform('mean')
        transactions_frame['max_utilization_ratio'] = grp['utilization_ratio'].transform('max')

        # last payment
        transactions_frame['last_paym_status'] = grp['enc_paym_0'].transform('last')

        # bad loan ratio
        transactions_frame["is_bad_loan"] = (transactions_frame["paym_bad_count"] > 0).astype(int)
        transactions_frame["client_bad_loan_ratio"] = grp["is_bad_loan"].transform("mean")

        # credit structure
        transactions_frame["client_total_credit"] = grp["pre_loans_credit_limit"].transform("sum")
        transactions_frame["max_credit_share"] = (
            grp["pre_loans_credit_limit"].transform("max") /
            (transactions_frame["client_total_credit"] + 1e-6)
        )

        # utilization volatility
        transactions_frame["util_ratio"] = (
            transactions_frame["pre_loans_outstanding"] /
            (transactions_frame["pre_loans_credit_limit"] + 1e-6)
        )
        transactions_frame["client_util_std"] = grp["util_ratio"].transform("std").fillna(0)

        transactions_frame["bad_loan_weight"] = (
            transactions_frame["is_bad_loan"] * transactions_frame["pre_loans_outstanding"]
        )
        transactions_frame["client_bad_weight_share"] = (
            grp["bad_loan_weight"].transform("sum") /
            (grp["pre_loans_outstanding"].transform("sum") + 1e-6)
        )

        # worst loan
        transactions_frame["loan_risk_score"] = transactions_frame["paym_bad_count"]
        transactions_frame["client_worst_loan"] = grp["loan_risk_score"].transform("max")

        # severe loans
        transactions_frame["is_severe_loan"] = (transactions_frame["paym_severe_count"] > 0).astype(int)
        transactions_frame["client_severe_loan_ratio"] = grp["is_severe_loan"].transform("mean")

        # credit diversity
        transactions_frame["credit_type_count"] = grp["enc_loans_credit_type"].transform("nunique")
        transactions_frame["credit_type_density"] = (
            transactions_frame["credit_type_count"] /
            (transactions_frame["client_products_count"] + 1)
        )

        # decay payments
        weights = np.array([0.9**i for i in range(24)])
        transactions_frame["decayed_paym_score"] = (
            transactions_frame[paym_cols].values * weights
        ).sum(axis=1)

        # recent vs total
        recent_cols = [f"enc_paym_{i}" for i in range(12, 24)]

        transactions_frame["recent_bad_count"] = (
            transactions_frame[recent_cols] > 0
        ).sum(axis=1)

        transactions_frame["recent_bad_ratio"] = (
            grp["recent_bad_count"].transform("sum") /
            (grp["paym_bad_count"].transform("sum") + 1e-6)
        )

        # --- базовые per-loan фичи ---
        transactions_frame["loan_paym_mean"] = transactions_frame[paym_cols].mean(axis=1)
        transactions_frame["loan_paym_max"] = transactions_frame[paym_cols].max(axis=1)

        transactions_frame["loan_util"] = (
            transactions_frame["pre_loans_outstanding"] /
            (transactions_frame["pre_loans_credit_limit"] + 1e-6)
        )

        transactions_frame["loan_overdue"] = transactions_frame["pre_loans_max_overdue_sum"]

        # --- WORST LOAN ---
        # индекс худшего кредита
        idx_worst = transactions_frame.groupby("id")["paym_bad_count"].idxmax()

        worst_df = transactions_frame.loc[idx_worst, [
            "id",
            "loan_paym_mean",
            "loan_paym_max",
            "loan_util",
            "loan_overdue",
            "pre_loans_outstanding",
            "pre_loans_credit_limit"
        ]].copy()

        worst_df = worst_df.rename(columns={
            "loan_paym_mean": "worst_loan_paym_mean",
            "loan_paym_max": "worst_loan_paym_max",
            "loan_util": "worst_loan_util",
            "loan_overdue": "worst_loan_overdue",
            "pre_loans_outstanding": "worst_loan_outstanding",
            "pre_loans_credit_limit": "worst_loan_credit_limit"
        })

        # нормализация
        worst_df["worst_loan_overdue_to_limit"] = (
            worst_df["worst_loan_overdue"] /
            (worst_df["worst_loan_credit_limit"] + 1e-6)
        )

        # --- LAST LOAN ---
        idx_last = transactions_frame.groupby("id")["rn"].idxmax()

        last_df = transactions_frame.loc[idx_last, [
            "id",
            "loan_paym_mean",
            "loan_paym_max",
            "loan_util",
            "loan_overdue",
            "pre_loans_outstanding"
        ]].copy()

        last_df = last_df.rename(columns={
            "loan_paym_mean": "last_loan_paym_mean",
            "loan_paym_max": "last_loan_paym_max",
            "loan_util": "last_loan_util",
            "loan_overdue": "last_loan_overdue",
            "pre_loans_outstanding": "last_loan_outstanding"
        })

        # --- DISPERSION ---
        disp_df = transactions_frame.groupby("id").agg({
            "loan_paym_mean": ["std", "max", "mean"],
            "loan_util": ["std", "max", "mean"],
            "loan_overdue": ["std", "max", "mean"]
        })

        disp_df.columns = [
            "loan_paym_std",
            "loan_paym_max",
            "loan_paym_mean_global",
            "loan_util_std",
            "loan_util_max",
            "loan_util_mean",
            "loan_overdue_std",
            "loan_overdue_max",
            "loan_overdue_mean"
        ]

        disp_df = disp_df.reset_index()

        # max - mean
        disp_df["loan_paym_max_minus_mean"] = (
            disp_df["loan_paym_max"] - disp_df["loan_paym_mean_global"]
        )

        disp_df["loan_util_max_minus_mean"] = (
            disp_df["loan_util_max"] - disp_df["loan_util_mean"]
        )

        disp_df["loan_overdue_max_minus_mean"] = (
            disp_df["loan_overdue_max"] - disp_df["loan_overdue_mean"]
        )


        df = transactions_frame.groupby("id").agg({
            "pre_since_opened": ["min", "max", "mean"],
            "pre_since_confirmed": ["min", "max", "mean"],
            "pre_pterm": ["min", "max", "mean"],
            "pre_fterm": ["min", "max", "mean"],
            "pre_till_pclose": ["min", "max"],
            "pre_till_fclose": ["min", "max"],

            "pre_loans_credit_limit": ["max", "mean"],
            "pre_loans_next_pay_summ": ["mean", "max"],
            "pre_loans_outstanding": ["max", "mean"],
            "pre_loans_max_overdue_sum": ["max"],
            "pre_loans_credit_cost_rate": ["mean"],

            "pre_loans5": ["sum"],
            "pre_loans530": ["sum", "max"],
            "pre_loans3060": ["max"],

            "is_zero_loans5": ["mean", "sum"],
            "is_zero_loans530": ["mean", "sum"],
            "is_zero_loans3060": ["mean"],
            "is_zero_loans6090": ["mean"],
            "is_zero_loans90": ["mean", "min"],

            "pre_util": ["mean", "max"],
            "pre_over2limit": ["max"],
            "pre_maxover2limit": ["max"],

            "is_zero_util": ["mean"],
            "is_zero_over2limit": ["mean"],
            "is_zero_maxover2limit": ["mean"],

            "enc_loans_account_holder_type": ["nunique"],
            "enc_loans_credit_status": ["nunique"],
            "enc_loans_credit_type": ["nunique"],
            "enc_loans_account_cur": ["nunique"],

            "pclose_flag": ["mean"],
            "fclose_flag": ["mean"],

            "enc_paym_0": ["max", "mean", "sum"],
            "enc_paym_1": ["max", "mean", "sum"],
            "enc_paym_2": ["max", "mean", "sum"],
            "enc_paym_3": ["max", "mean", "sum"],
            "enc_paym_4": ["max", "mean", "sum"],
            "enc_paym_5": ["max", "mean", "sum"],
            "enc_paym_6": ["max", "mean", "sum"],
            "enc_paym_7": ["mean", "sum"],
            "enc_paym_8": ["mean", "sum"],
            "enc_paym_9": ["mean", "sum"],
            "enc_paym_10": ["mean", "sum"],
            "enc_paym_11": ["mean", "sum"],
            "enc_paym_12": ["mean", "sum"],
            "enc_paym_13": ["mean", "sum"],
            "enc_paym_14": ["mean", "sum"],
            "enc_paym_15": ["mean", "sum"],
            "enc_paym_16": ["mean", "sum"],
            "enc_paym_17": ["mean", "sum"],
            "enc_paym_18": ["mean", "sum"],
            "enc_paym_19": ["mean", "sum"],
            "enc_paym_20": ["mean", "sum"],
            "enc_paym_21": ["mean", "sum"],
            "enc_paym_22": ["mean", "sum"],
            "enc_paym_23": ["mean", "sum"],

            "paym_bad_count": ["mean"],
            "paym_severe_count": ["mean"],
            "paym_nonzero_share": ["mean"],
            "paym_mean_status": ["mean"],
            "enc_paym_weighted_score": ["mean", "max"],

            "client_products_count": "max",
            "mean_utilization_ratio": "max",
            "max_utilization_ratio": "max",
            "last_paym_status": "max",
            "client_bad_loan_ratio": "max",
            "client_util_std": "max",
            "max_credit_share": "max",
            "recent_bad_ratio": "max",
            "client_worst_loan": "max",
            "decayed_paym_score": ["mean", "max"],

            "last_vs_prev_util": "last",
            "last_vs_prev_paym": "last",
            "util_rank": "last",
            "paym_rank": "last",
            "last_vs_last3_util": "last",
            "last_vs_last3_paym": "last",
        }).reset_index()

        df.columns = [
            '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
            for col in df.columns
        ]

        client_counts = transactions_frame.groupby("id").size().rename("client_products_count")
        df = df.merge(client_counts, on="id", how="left")

        df = df.rename(columns={
            "paym_bad_count_mean": "client_mean_bad_months_per_product",
            "paym_nonzero_share_mean": "client_mean_nonzero_paym_share",
            "enc_paym_weighted_score_mean": "client_mean_weighted_paym_score",
            "paym_mean_status_mean": "client_mean_paym_status",
            "enc_paym_weighted_score_max": "client_max_weighted_paym_score",
            "paym_severe_count_mean": "client_mean_severe_bad_months",
        })


        df = df.merge(worst_df, on="id", how="left")
        df = df.merge(last_df, on="id", how="left")
        df = df.merge(disp_df, on="id", how="left")


        # --- дополнительные отношения (уже на уровне df) ---
        df["last_vs_mean_util"] = (
            df["last_loan_util"] - df["pre_util_mean"]
        )

        df["last_vs_mean_paym"] = (
            df["last_loan_paym_mean"] - df["client_mean_paym_status"]
        )

        df["last_vs_worst_paym"] = (
            df["last_loan_paym_mean"] - df["worst_loan_paym_mean"]
        )

        df["worst_loan_util_to_mean_util"] = (
            df["worst_loan_util"] / (df["pre_util_mean"] + 1e-6)
        )

        df["max_credit_share"] = transactions_frame.groupby("id")["max_credit_share"].max().values

        # 1. Нагрузка на один кредит
        df["util_per_loan"] = (
            df["pre_util_mean"] / (df["pre_loans_outstanding_mean"] + 1)
        )

        # 2. Нестабильность utilization
        df["util_stability"] = (
            df["pre_util_max"] - df["pre_util_mean"]
        )

        # 3. Regression trend
        cols = [f"enc_paym_{i}_mean" for i in range(13)]

        X_vals = df[cols].values
        x = np.arange(len(cols))

        x_mean = x.mean()
        denominator = ((x - x_mean) ** 2).sum()

        df["paym_trend_lr"] = (
            ((X_vals - X_vals.mean(axis=1, keepdims=True)) * (x - x_mean))
            .sum(axis=1) / denominator
        )

        # 4. Bad total
        df["bad_to_total_ratio"] = (
            df["client_mean_bad_months_per_product"] /
            (df["client_mean_paym_status"] + 1)
        )

        # 5.
        df["paym_trend_abs"] = np.abs(df["paym_trend_lr"])

        # 6. last bad month
        cols = [f"enc_paym_{i}_mean" for i in range(13)]
        mask = (df[cols[::-1]] > 0).values
        has_bad = mask.sum(axis=1) > 0

        last_idx = mask.argmax(axis=1)
        df["last_bad_month"] = np.where(has_bad, last_idx, -1)

        # 7. recent vs old
        recent = [f"enc_paym_{i}_mean" for i in range(12, 24)]
        old = [f"enc_paym_{i}_mean" for i in range(0, 12)]

        df["recent_vs_old"] = df[recent].mean(axis=1) - df[old].mean(axis=1)

        # 8. recent vs old (early)
        recent = [f"enc_paym_{i}_mean" for i in range(6, 13)]
        old = [f"enc_paym_{i}_mean" for i in range(0, 5)]

        df["recent_vs_old_early"] = df[recent].mean(axis=1) - df[old].mean(axis=1)

        # 9. Стабильность клиента
        cols = [f"enc_paym_{i}_mean" for i in range(13)]
        
        df["paym_std"] = df[cols].std(axis=1)

        # 10. самый худший месяц
        cols = [f"enc_paym_{i}_max" for i in range(7)] + [f"enc_paym_{i}_mean" for i in range(7, 24)]

        df["worst_paym_ever"] = df[cols].max(axis=1)

        # 11. Нестабильность клиента
        cols = [f"enc_paym_{i}_mean" for i in range(13)]

        arr = df[cols].values
        df["paym_switches"] = (np.diff(arr > 0, axis=1) != 0).sum(axis=1)

        # 12. Текущее состояние
        df["last_vs_mean"] = (
            df["last_paym_status_max"] - df["client_mean_paym_status"]
        )

        eps = 1e-6

        df["last_vs_std_util"] = (
            (df["last_loan_util"] - df["pre_util_mean"]) /
            (df["loan_util_std"] + eps)
        )

        df["last_vs_std_paym"] = (
            (df["last_loan_paym_mean"] - df["client_mean_paym_status"]) /
            (df["loan_paym_std"] + eps)
        )

        df["last_util_x_std"] = df["last_loan_util"] * df["loan_util_std"]
        df["last_paym_x_std"] = df["last_loan_paym_mean"] * df["loan_paym_std"]

        df["last_vs_mean_util_x_std"] = (
            (df["last_loan_util"] - df["pre_util_mean"]) * df["loan_util_std"]
        )

        df["last_vs_mean_paym_x_std"] = (
            (df["last_loan_paym_mean"] - df["client_mean_paym_status"]) * df["loan_paym_std"]
        )


        drop_cols = [
            "client_bad_loan_ratio_max",
            "last_bad_month",

            "enc_paym_2_max",
            "enc_paym_3_max",
            "enc_paym_4_max",
            "enc_paym_5_max",
            "enc_paym_6_max",

            "pre_loans530_max",
            "pre_loans3060_max",
            "pre_maxover2limit_max",

            "enc_paym_12_sum",
            "enc_paym_13_sum",
            "enc_paym_14_sum",
            "enc_paym_15_sum",
            "enc_paym_16_sum",
            "enc_paym_17_sum",
            "enc_paym_18_sum"
        ]

        existing_cols_to_drop = [col for col in drop_cols if col in df.columns]
        df = df.drop(columns=existing_cols_to_drop)

        
        display(df)
        df.info(max_cols=5, memory_usage='deep')


        # записываем подготовленные данные в файл
        if save_to_path:
            block_as_str = str(step)
            if len(block_as_str) == 1:
                block_as_str = '00' + block_as_str
            else:
                block_as_str = '0' + block_as_str
            df.to_parquet(os.path.join(save_to_path, f'processed_chunk_{block_as_str}.parquet'))
        gc.collect()
        preprocessed_frames.append(df)

    return pd.concat(preprocessed_frames)

## pipeline old version

In [1]:
def prepare_transactions_dataset(path_to_dataset: str, file_name_tag: str, num_parts_to_preprocess_at_once: int = 1, num_parts_total: int=50,
                                 save_to_path=None, verbose: bool=False):
    """
    возвращает готовый pd.DataFrame с признаками, на которых можно учить модель для целевой задачи.
    path_to_dataset: str
        путь до датасета с партициями
    num_parts_to_preprocess_at_once: int
        количество партиций, которые будут одновременно держаться в памяти и обрабатываться
    num_parts_total: int
        общее количество партиций, которые нужно обработать
    save_to_path: str
        путь до папки, в которой будет сохранен каждый обработанный блок в .parquet формате. Если None, то не будет сохранен
    verbose: bool
        логирует каждый обрабатываемый кусок данных
    """
    preprocessed_frames = []

    for step in tqdm(range(0, num_parts_total, num_parts_to_preprocess_at_once),
                               desc="Transforming transactions data"):
        transactions_frame = read_parquet_dataset_from_local(
            path_to_dataset, file_name_tag, step, num_parts_to_preprocess_at_once,
            verbose=verbose
        )

        # === базовые колонки ===
        paym_cols = [f"enc_paym_{i}" for i in range(24)]  # FIX: используем все 0..23

        # === платежные фичи ===
        transactions_frame["paym_bad_count"] = (transactions_frame[paym_cols] > 0).sum(axis=1)
        transactions_frame["paym_severe_count"] = (transactions_frame[paym_cols] >= 2).sum(axis=1)
        transactions_frame["paym_nonzero_share"] = (transactions_frame[paym_cols] > 0).mean(axis=1)
        transactions_frame["paym_mean_status"] = transactions_frame[paym_cols].mean(axis=1)
        transactions_frame["enc_paym_weighted_score"] = transactions_frame[paym_cols].sum(axis=1)

        # === сортировка (ВАЖНО для last) ===
        transactions_frame = transactions_frame.sort_values(['id', 'rn'])

        # === клиентские агрегаты (до groupby) ===
        grp = transactions_frame.groupby('id')

        transactions_frame['client_products_count'] = grp['id'].transform('count')

        # utilization
        transactions_frame['utilization_ratio'] = (
            transactions_frame['pre_loans_outstanding'] /
            (transactions_frame['pre_loans_credit_limit'] + 1e-6)
        )

        transactions_frame['mean_utilization_ratio'] = grp['utilization_ratio'].transform('mean')
        transactions_frame['max_utilization_ratio'] = grp['utilization_ratio'].transform('max')

        # last payment
        transactions_frame['last_paym_status'] = grp['enc_paym_0'].transform('last')

        # bad loan ratio
        transactions_frame["is_bad_loan"] = (transactions_frame["paym_bad_count"] > 0).astype(int)
        transactions_frame["client_bad_loan_ratio"] = grp["is_bad_loan"].transform("mean")

        # credit structure
        transactions_frame["client_total_credit"] = grp["pre_loans_credit_limit"].transform("sum")
        transactions_frame["max_credit_share"] = (
            grp["pre_loans_credit_limit"].transform("max") /
            (transactions_frame["client_total_credit"] + 1e-6)
        )

        # utilization volatility
        transactions_frame["util_ratio"] = (
            transactions_frame["pre_loans_outstanding"] /
            (transactions_frame["pre_loans_credit_limit"] + 1e-6)
        )
        transactions_frame["client_util_std"] = grp["util_ratio"].transform("std").fillna(0)

        transactions_frame["bad_loan_weight"] = (
            transactions_frame["is_bad_loan"] * transactions_frame["pre_loans_outstanding"]
        )
        transactions_frame["client_bad_weight_share"] = (
            grp["bad_loan_weight"].transform("sum") /
            (grp["pre_loans_outstanding"].transform("sum") + 1e-6)
        )

        # worst loan
        transactions_frame["loan_risk_score"] = transactions_frame["paym_bad_count"]
        transactions_frame["client_worst_loan"] = grp["loan_risk_score"].transform("max")

        # severe loans
        transactions_frame["is_severe_loan"] = (transactions_frame["paym_severe_count"] > 0).astype(int)
        transactions_frame["client_severe_loan_ratio"] = grp["is_severe_loan"].transform("mean")

        # credit diversity
        transactions_frame["credit_type_count"] = grp["enc_loans_credit_type"].transform("nunique")
        transactions_frame["credit_type_density"] = (
            transactions_frame["credit_type_count"] /
            (transactions_frame["client_products_count"] + 1)
        )

        # decay payments
        weights = np.array([0.9**i for i in range(24)])
        transactions_frame["decayed_paym_score"] = (
            transactions_frame[paym_cols].values * weights
        ).sum(axis=1)

        # recent vs total
        recent_cols = [f"enc_paym_{i}" for i in range(12, 24)]

        transactions_frame["recent_bad_count"] = (
            transactions_frame[recent_cols] > 0
        ).sum(axis=1)

        transactions_frame["recent_bad_ratio"] = (
            grp["recent_bad_count"].transform("sum") /
            (grp["paym_bad_count"].transform("sum") + 1e-6)
        )

        # --- базовые per-loan фичи ---
        transactions_frame["loan_paym_mean"] = transactions_frame[paym_cols].mean(axis=1)
        transactions_frame["loan_paym_max"] = transactions_frame[paym_cols].max(axis=1)

        transactions_frame["loan_util"] = (
            transactions_frame["pre_loans_outstanding"] /
            (transactions_frame["pre_loans_credit_limit"] + 1e-6)
        )

        transactions_frame["loan_overdue"] = transactions_frame["pre_loans_max_overdue_sum"]

        # --- WORST LOAN ---
        # индекс худшего кредита
        idx_worst = transactions_frame.groupby("id")["paym_bad_count"].idxmax()

        worst_df = transactions_frame.loc[idx_worst, [
            "id",
            "loan_paym_mean",
            "loan_paym_max",
            "loan_util",
            "loan_overdue",
            "pre_loans_outstanding",
            "pre_loans_credit_limit"
        ]].copy()

        worst_df = worst_df.rename(columns={
            "loan_paym_mean": "worst_loan_paym_mean",
            "loan_paym_max": "worst_loan_paym_max",
            "loan_util": "worst_loan_util",
            "loan_overdue": "worst_loan_overdue",
            "pre_loans_outstanding": "worst_loan_outstanding",
            "pre_loans_credit_limit": "worst_loan_credit_limit"
        })

        # нормализация
        worst_df["worst_loan_overdue_to_limit"] = (
            worst_df["worst_loan_overdue"] /
            (worst_df["worst_loan_credit_limit"] + 1e-6)
        )

        # --- LAST LOAN ---
        idx_last = transactions_frame.groupby("id")["rn"].idxmax()

        last_df = transactions_frame.loc[idx_last, [
            "id",
            "loan_paym_mean",
            "loan_paym_max",
            "loan_util",
            "loan_overdue",
            "pre_loans_outstanding"
        ]].copy()

        last_df = last_df.rename(columns={
            "loan_paym_mean": "last_loan_paym_mean",
            "loan_paym_max": "last_loan_paym_max",
            "loan_util": "last_loan_util",
            "loan_overdue": "last_loan_overdue",
            "pre_loans_outstanding": "last_loan_outstanding"
        })

        # --- DISPERSION ---
        disp_df = transactions_frame.groupby("id").agg({
            "loan_paym_mean": ["std", "max", "mean"],
            "loan_util": ["std", "max", "mean"],
            "loan_overdue": ["std", "max", "mean"]
        })

        disp_df.columns = [
            "loan_paym_std",
            "loan_paym_max",
            "loan_paym_mean_global",
            "loan_util_std",
            "loan_util_max",
            "loan_util_mean",
            "loan_overdue_std",
            "loan_overdue_max",
            "loan_overdue_mean"
        ]

        disp_df = disp_df.reset_index()

        # max - mean
        disp_df["loan_paym_max_minus_mean"] = (
            disp_df["loan_paym_max"] - disp_df["loan_paym_mean_global"]
        )

        disp_df["loan_util_max_minus_mean"] = (
            disp_df["loan_util_max"] - disp_df["loan_util_mean"]
        )

        disp_df["loan_overdue_max_minus_mean"] = (
            disp_df["loan_overdue_max"] - disp_df["loan_overdue_mean"]
        )


        df = transactions_frame.groupby("id").agg({
            "pre_since_opened": ["min", "max", "mean"],
            "pre_since_confirmed": ["min", "max", "mean"],
            "pre_pterm": ["min", "max", "mean"],
            "pre_fterm": ["min", "max", "mean"],
            "pre_till_pclose": ["min", "max"],
            "pre_till_fclose": ["min", "max"],

            "pre_loans_credit_limit": ["max", "mean"],
            "pre_loans_next_pay_summ": ["mean", "max"],
            "pre_loans_outstanding": ["max", "mean"],
            "pre_loans_max_overdue_sum": ["max"],
            "pre_loans_credit_cost_rate": ["mean"],

            "pre_loans5": ["sum"],
            "pre_loans530": ["sum", "max"],
            "pre_loans3060": ["max"],

            "is_zero_loans5": ["mean", "sum"],
            "is_zero_loans530": ["mean", "sum"],
            "is_zero_loans3060": ["mean"],
            "is_zero_loans6090": ["mean"],
            "is_zero_loans90": ["mean"],

            "pre_util": ["mean", "max"],
            "pre_over2limit": ["max"],
            "pre_maxover2limit": ["max"],

            "is_zero_util": ["mean"],
            "is_zero_over2limit": ["mean"],
            "is_zero_maxover2limit": ["mean"],

            "enc_loans_account_holder_type": ["nunique"],
            "enc_loans_credit_status": ["nunique"],
            "enc_loans_credit_type": ["nunique"],
            "enc_loans_account_cur": ["nunique"],

            "pclose_flag": ["mean"],
            "fclose_flag": ["mean"],

            "enc_paym_0": ["max", "mean", "sum"],
            "enc_paym_1": ["max", "mean", "sum"],
            "enc_paym_2": ["max", "mean", "sum"],
            "enc_paym_3": ["max", "mean", "sum"],
            "enc_paym_4": ["max", "mean", "sum"],
            "enc_paym_5": ["max", "mean", "sum"],
            "enc_paym_6": ["max", "mean", "sum"],
            "enc_paym_7": ["mean", "sum"],
            "enc_paym_8": ["mean", "sum"],
            "enc_paym_9": ["mean", "sum"],
            "enc_paym_10": ["mean", "sum"],
            "enc_paym_11": ["mean", "sum"],
            "enc_paym_12": ["mean", "sum"],
            "enc_paym_13": ["mean", "sum"],
            "enc_paym_14": ["mean", "sum"],
            "enc_paym_15": ["mean", "sum"],
            "enc_paym_16": ["mean", "sum"],
            "enc_paym_17": ["mean", "sum"],
            "enc_paym_18": ["mean", "sum"],
            "enc_paym_19": ["mean", "sum"],
            "enc_paym_20": ["mean", "sum"],
            "enc_paym_21": ["mean", "sum"],
            "enc_paym_22": ["mean", "sum"],
            "enc_paym_23": ["mean", "sum"],

            "paym_bad_count": ["mean"],
            "paym_severe_count": ["mean"],
            "paym_nonzero_share": ["mean"],
            "paym_mean_status": ["mean"],
            "enc_paym_weighted_score": ["mean", "max"],

            "client_products_count": "max",
            "mean_utilization_ratio": "max",
            "max_utilization_ratio": "max",
            "last_paym_status": "max",
            "client_bad_loan_ratio": "max",
            "client_util_std": "max",
            "max_credit_share": "max",
            "recent_bad_ratio": "max",
            "client_worst_loan": "max",
            "decayed_paym_score": ["mean", "max"],
        }).reset_index()

        df.columns = [
            '_'.join(col).rstrip('_') if isinstance(col, tuple) else col
            for col in df.columns
        ]

        client_counts = transactions_frame.groupby("id").size().rename("client_products_count")
        df = df.merge(client_counts, on="id", how="left")

        df = df.rename(columns={
            "paym_bad_count_mean": "client_mean_bad_months_per_product",
            "paym_nonzero_share_mean": "client_mean_nonzero_paym_share",
            "enc_paym_weighted_score_mean": "client_mean_weighted_paym_score",
            "paym_mean_status_mean": "client_mean_paym_status",
            "enc_paym_weighted_score_max": "client_max_weighted_paym_score",
            "paym_severe_count_mean": "client_mean_severe_bad_months",
        })


        df = df.merge(worst_df, on="id", how="left")
        df = df.merge(last_df, on="id", how="left")
        df = df.merge(disp_df, on="id", how="left")

        # --- дополнительные отношения (уже на уровне df) ---
        df["last_vs_mean_util"] = (
            df["last_loan_util"] - df["pre_util_mean"]
        )

        df["last_vs_mean_paym"] = (
            df["last_loan_paym_mean"] - df["client_mean_paym_status"]
        )

        df["last_vs_worst_paym"] = (
            df["last_loan_paym_mean"] - df["worst_loan_paym_mean"]
        )

        df["worst_loan_util_to_mean_util"] = (
            df["worst_loan_util"] / (df["pre_util_mean"] + 1e-6)
        )

        df["max_credit_share"] = transactions_frame.groupby("id")["max_credit_share"].max().values

        # 1. Нагрузка на один кредит
        df["util_per_loan"] = (
            df["pre_util_mean"] / (df["pre_loans_outstanding_mean"] + 1)
        )

        # 2. Нестабильность utilization
        df["util_stability"] = (
            df["pre_util_max"] - df["pre_util_mean"]
        )

        # 3. Regression trend
        cols = [f"enc_paym_{i}_mean" for i in range(13)]

        X_vals = df[cols].values
        x = np.arange(len(cols))

        x_mean = x.mean()
        denominator = ((x - x_mean) ** 2).sum()

        df["paym_trend_lr"] = (
            ((X_vals - X_vals.mean(axis=1, keepdims=True)) * (x - x_mean))
            .sum(axis=1) / denominator
        )

        # 4. Bad total
        df["bad_to_total_ratio"] = (
            df["client_mean_bad_months_per_product"] /
            (df["client_mean_paym_status"] + 1)
        )

        # 5.
        df["paym_trend_abs"] = np.abs(df["paym_trend_lr"])

        # 6. last bad month
        cols = [f"enc_paym_{i}_mean" for i in range(13)]
        mask = (df[cols[::-1]] > 0).values
        has_bad = mask.sum(axis=1) > 0

        last_idx = mask.argmax(axis=1)
        df["last_bad_month"] = np.where(has_bad, last_idx, -1)

        # 7. recent vs old
        recent = [f"enc_paym_{i}_mean" for i in range(12, 24)]
        old = [f"enc_paym_{i}_mean" for i in range(0, 12)]

        df["recent_vs_old"] = df[recent].mean(axis=1) - df[old].mean(axis=1)

        # 8. recent vs old (early)
        recent = [f"enc_paym_{i}_mean" for i in range(6, 13)]
        old = [f"enc_paym_{i}_mean" for i in range(0, 5)]

        df["recent_vs_old_early"] = df[recent].mean(axis=1) - df[old].mean(axis=1)

        # 9. Стабильность клиента
        cols = [f"enc_paym_{i}_mean" for i in range(13)]
        
        df["paym_std"] = df[cols].std(axis=1)

        # 10. самый худший месяц
        cols = [f"enc_paym_{i}_max" for i in range(7)] + [f"enc_paym_{i}_mean" for i in range(7, 24)]

        df["worst_paym_ever"] = df[cols].max(axis=1)

        # 11. Нестабильность клиента
        cols = [f"enc_paym_{i}_mean" for i in range(13)]

        arr = df[cols].values
        df["paym_switches"] = (np.diff(arr > 0, axis=1) != 0).sum(axis=1)

        # 12. Текущее состояние
        df["last_vs_mean"] = (
            df["last_paym_status_max"] - df["client_mean_paym_status"]
        )


        drop_cols = [
            "client_bad_loan_ratio_max",
            "last_bad_month",

            "enc_paym_2_max",
            "enc_paym_3_max",
            "enc_paym_4_max",
            "enc_paym_5_max",
            "enc_paym_6_max",

            "pre_loans530_max",
            "pre_loans3060_max",
            "pre_maxover2limit_max",

            "enc_paym_12_sum",
            "enc_paym_13_sum",
            "enc_paym_14_sum",
            "enc_paym_15_sum",
            "enc_paym_16_sum",
            "enc_paym_17_sum",
            "enc_paym_18_sum"
        ]

        existing_cols_to_drop = [col for col in drop_cols if col in df.columns]
        df = df.drop(columns=existing_cols_to_drop)

        
        display(df)
        df.info(max_cols=5, memory_usage='deep')


        # записываем подготовленные данные в файл
        if save_to_path:
            block_as_str = str(step)
            if len(block_as_str) == 1:
                block_as_str = '00' + block_as_str
            else:
                block_as_str = '0' + block_as_str
            df.to_parquet(os.path.join(save_to_path, f'processed_chunk_{block_as_str}.parquet'))
        gc.collect()
        preprocessed_frames.append(df)

    return pd.concat(preprocessed_frames)

# 3. Dataset Construction

In this section:
- raw data is processed using the pipeline functions
- multiple data sources are combined
- final dataset is constructed and merged with target

## preprocessing

In [10]:
data = prepare_transactions_dataset(path, 'train', num_parts_to_preprocess_at_once=1, num_parts_total=12,
                                    save_to_path=path)

Reading dataset with pandas:   0%|                       | 0/1 [00:00<?, ?it/s]

chunk_path C:\Users\User\-_PRACTICAL\junior-final-work\train_data\train_data_0.pq
Memory usage of dataframe is 919.02 MB




  0%|                                                   | 0/61 [00:00<?, ?it/s]

 30%|████████████                             | 18/61 [00:00<00:00, 172.71it/s]

 59%|████████████████████████▏                | 36/61 [00:00<00:00, 173.73it/s]

100%|█████████████████████████████████████████| 61/61 [00:00<00:00, 158.66it/s]

Reading dataset with pandas: 100%|███████████████| 1/1 [00:01<00:00,  1.17s/it]


Memory usage after optimization is: 120.528 MB
Decreased by 86.9%


,id,pre_since_opened_min,pre_since_opened_max,pre_since_opened_mean,pre_since_confirmed_min,pre_since_confirmed_max,pre_since_confirmed_mean,pre_pterm_min,pre_pterm_max,pre_pterm_mean,...,paym_std,worst_paym_ever,paym_switches,last_vs_mean,last_vs_std_util,last_vs_std_paym,last_util_x_std,last_paym_x_std,last_vs_mean_util_x_std,last_vs_mean_paym_x_std
0,0,1,18,8.100000,0,12,7.600000,1,15,7.100000,...,0.714592,3.400000,2,-1.612500,-0.000014,0.696151,118585.391439,2.416059,-1.278351e+07,0.746390
1,1,2,18,11.428571,3,14,7.642857,0,15,6.642857,...,0.598435,3.142857,0,1.294643,-0.000015,1.433731,218668.211377,2.963427,-9.574545e+06,1.324389
2,2,0,13,8.333333,9,14,10.666667,4,13,7.000000,...,0.738637,3.333333,0,1.097222,-2.836837,1.028886,6.110088,3.537849,-6.619274e+00,1.354582
3,3,1,18,7.000000,0,16,7.333333,0,16,7.600000,...,0.615609,3.000000,1,-1.075000,-12.723924,1.835863,0.111715,3.034825,-1.147312e+01,1.932029
4,4,12,12,12.000000,9,9,9.000000,4,4,4.000000,...,0.277350,4.000000,0,-0.083333,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,249995,0,18,7.615385,1,14,7.538462,0,16,10.230769,...,1.086880,4.000000,3,-1.923077,-11.037512,0.686996,0.189212,0.916948,-1.585013e+01,0.118445
249996,249996,1,19,9.217391,0,17,9.782609,2,17,8.652174,...,0.843664,3.608696,0,-1.914855,-0.000023,1.139251,41702.874612,2.709634,-8.933482e+06,0.955756
249997,249997,1,14,6.857143,2,6,5.142857,1,17,10.428571,...,0.602845,3.000000,2,-0.958333,-15.835339,0.672534,0.489411,1.905105,-1.517175e+01,0.836388
249998,249998,1,15,6.800000,5,9,7.800000,4,14,9.600000,...,0.686126,4.000000,0,-2.333333,-46.456948,0.171498,0.054332,1.791803,-3.094196e+00,0.091109


...



  0%|                                                   | 0/61 [00:00<?, ?it/s]

 21%|████████▋                                | 13/61 [00:00<00:00, 129.32it/s]

 44%|██████████████████▏                      | 27/61 [00:00<00:00, 133.25it/s]

 67%|███████████████████████████▌             | 41/61 [00:00<00:00, 124.10it/s]

100%|█████████████████████████████████████████| 61/61 [00:00<00:00, 107.32it/s]

Reading dataset with pandas: 100%|███████████████| 1/1 [00:01<00:00,  1.48s/it]


Memory usage after optimization is: 139.420 MB
Decreased by 86.9%


,id,pre_since_opened_min,pre_since_opened_max,pre_since_opened_mean,pre_since_confirmed_min,pre_since_confirmed_max,pre_since_confirmed_mean,pre_pterm_min,pre_pterm_max,pre_pterm_mean,...,paym_std,worst_paym_ever,paym_switches,last_vs_mean,last_vs_std_util,last_vs_std_paym,last_util_x_std,last_paym_x_std,last_vs_mean_util_x_std,last_vs_mean_paym_x_std
0,2250000,2,12,7.250000,2,17,7.750000,4,16,10.375000,...,0.759291,4.000000,1,-2.213542,-15.831213,1.006257,2.207550e-01,2.189638,-1.448704e+01,0.551265
1,2250001,12,12,12.000000,2,16,9.000000,4,16,10.000000,...,0.898717,4.000000,1,-2.958333,-6.942507,0.000000,2.592719e+00,0.000000,-1.166725e+01,0.000000
2,2250002,2,16,8.333333,4,16,9.111111,0,17,10.333333,...,0.507463,3.000000,1,-0.972222,-0.000006,1.578887,1.375000e+06,2.636054,-1.940278e+07,1.643991
3,2250003,9,12,11.250000,0,10,6.500000,1,6,3.500000,...,0.732160,4.000000,0,-2.645833,-34.542018,0.296078,3.039765e-01,1.794289,-3.191755e+00,0.118740
4,2250004,3,3,3.000000,6,6,6.000000,2,2,2.000000,...,0.650444,2.000000,10,-0.375000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,2499995,0,19,12.642857,1,11,5.714286,0,15,7.214286,...,0.706135,3.571429,1,-1.994048,-0.000019,0.893849,4.952520e+05,2.660377,-1.282939e+07,0.788053
249996,2499996,1,11,3.400000,2,17,11.600000,8,14,12.200000,...,0.713604,3.400000,1,-1.483333,-144.970672,-0.037012,2.267675e-02,0.985050,-1.192797e+00,-0.016887
249997,2499997,1,17,10.714286,0,17,7.428571,0,17,10.285714,...,0.533420,3.000000,1,-1.127976,-19.919814,0.205950,2.612511e-01,1.329493,-1.065905e+01,0.204766
249998,2499998,3,19,10.142857,9,12,9.714286,1,14,4.142857,...,0.538181,3.000000,1,-0.851190,-250.133053,1.541495,1.729663e-02,2.902831,-9.167215e-01,1.946368


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Columns: 159 entries, id to last_vs_mean_paym_x_std
dtypes: float64(98), int32(1), int64(24), int8(36)
memory usage: 242.2 MB


Transforming transactions data: 100%|██████████| 12/12 [19:26<00:00, 97.17s/it]


In [11]:
data = read_parquet_dataset_from_local(path, 'processed', 0, 12, verbose=1)

Reading chunks:

C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_000.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_001.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_002.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_003.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_004.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_005.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_006.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_007.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_008.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_009.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_010.parquet
C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk

Reading dataset with pandas:   0%|                      | 0/12 [00:00<?, ?it/s]

chunk_path C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_000.parquet
Memory usage of dataframe is 235.56 MB



Reading dataset with pandas:   8%|█▏            | 1/12 [00:00<00:08,  1.26it/s]

Memory usage after optimization is: 72.479 MB
Decreased by 69.2%
chunk_path C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_001.parquet
Memory usage of dataframe is 242.23 MB


...

Memory usage after optimization is: 73.195 MB
Decreased by 69.8%
chunk_path C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_010.parquet
Memory usage of dataframe is 230.55 MB



Reading dataset with pandas:  92%|███████████▉ | 11/12 [00:08<00:00,  1.39it/s]

Memory usage after optimization is: 71.526 MB
Decreased by 69.0%
chunk_path C:\Users\User\-_PRACTICAL\junior-final-work\train_data\processed_chunk_011.parquet
Memory usage of dataframe is 242.23 MB



Reading dataset with pandas: 100%|█████████████| 12/12 [00:08<00:00,  1.37it/s]


Memory usage after optimization is: 73.195 MB
Decreased by 69.8%


In [12]:
data.info(max_cols=5, memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Columns: 159 entries, id to last_vs_mean_paym_x_std
dtypes: float16(81), float32(17), int16(13), int32(1), int8(45), uint8(2)
memory usage: 878.3 MB


## merge with target

In [13]:
target_path = r"C:\Users\User\-_PRACTICAL\junior-final-work\train_target.csv"
target_df = pd.read_csv(target_path)

data = data.merge(target_df, on='id', how='left')

print(data.shape)
print(data.head())

(3000000, 160)
   id  pre_since_opened_min  pre_since_opened_max  pre_since_opened_mean  \
0   0                     1                    18               8.101562   
1   1                     2                    18              11.429688   
2   2                     0                    13               8.335938   
3   3                     1                    18               7.000000   
4   4                    12                    12              12.000000   

   pre_since_confirmed_min  pre_since_confirmed_max  pre_since_confirmed_mean  \
0                        0                       12                  7.601562   
1                        3                       14                  7.644531   
2                        9                       14                 10.664062   
3                        0                       16                  7.332031   
4                        9                        9                  9.000000   

   pre_pterm_min  pre_pterm_max  pre_pter

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [15]:
X = data.drop(['flag', 'id'], axis=1)
y = data["flag"]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}
print("Class weights:", class_weights_dict)

Class weights: {0: np.float64(0.5183929266321947), 1: np.float64(14.092181657616354)}


# 4. Baseline Models & Experiments

This section contains multiple experiments with different models and configurations.
The goal is to explore model behavior and establish strong baselines.

## modeling 13.03

In [160]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    train_size=0.7, 
    stratify=y, 
    random_state=42
)

In [51]:
model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=100
)

model.fit(X_train, y_train,
          eval_set=(X_val, y_val))

0:	test: 0.5898612	best: 0.5898612 (0)	total: 300ms	remaining: 4m 59s
100:	test: 0.7037026	best: 0.7037026 (100)	total: 32.3s	remaining: 4m 47s
200:	test: 0.7120052	best: 0.7120052 (200)	total: 1m 5s	remaining: 4m 18s
300:	test: 0.7165839	best: 0.7165839 (300)	total: 1m 37s	remaining: 3m 46s
400:	test: 0.7198728	best: 0.7198728 (400)	total: 2m 10s	remaining: 3m 14s
500:	test: 0.7222433	best: 0.7222433 (500)	total: 2m 42s	remaining: 2m 42s
600:	test: 0.7242320	best: 0.7242320 (600)	total: 3m 20s	remaining: 2m 13s
700:	test: 0.7256523	best: 0.7256523 (700)	total: 3m 55s	remaining: 1m 40s
800:	test: 0.7267832	best: 0.7267832 (800)	total: 4m 30s	remaining: 1m 7s
900:	test: 0.7276708	best: 0.7276708 (900)	total: 5m 5s	remaining: 33.6s
999:	test: 0.7283825	best: 0.7283825 (999)	total: 5m 39s	remaining: 0us

bestTest = 0.7283824774
bestIteration = 999



## param grid

In [60]:
grid = {
    'depth': [6, 8],
    'learning_rate': [0.03, 0.05],
    'l2_leaf_reg': [5, 7],
}

best_auc = 0
best_params = None

for depth, lr, l2 in product(grid['depth'], grid['learning_rate'], grid['l2_leaf_reg']):
    print(f"Testing: depth={depth}, lr={lr}, l2_leaf_reg={l2}")
    model = CatBoostClassifier(
        iterations=1000,
        depth=depth,
        learning_rate=lr,
        l2_leaf_reg=l2,
        loss_function='Logloss',
        eval_metric='AUC',
        early_stopping_rounds=50,
        verbose=100
    )
    model.fit(X_train, y_train, eval_set=(X_val, y_val))
    
    auc = model.get_best_score()['validation']['AUC']
    print(f"AUC: {auc}\n")
    
    if auc > best_auc:
        best_auc = auc
        best_params = {'depth': depth, 'learning_rate': lr, 'l2_leaf_reg': l2}

print(f"Best AUC: {best_auc} with params {best_params}")

Testing: depth=6, lr=0.03, l2_leaf_reg=5
0:	test: 0.5594944	best: 0.5594944 (0)	total: 325ms	remaining: 5m 24s
100:	test: 0.7034215	best: 0.7034215 (100)	total: 33.3s	remaining: 4m 56s
200:	test: 0.7113022	best: 0.7113022 (200)	total: 1m 14s	remaining: 4m 56s
300:	test: 0.7159701	best: 0.7159701 (300)	total: 1m 55s	remaining: 4m 27s
400:	test: 0.7189414	best: 0.7189414 (400)	total: 2m 35s	remaining: 3m 52s
500:	test: 0.7213733	best: 0.7213733 (500)	total: 3m 13s	remaining: 3m 12s
600:	test: 0.7231458	best: 0.7231458 (600)	total: 3m 49s	remaining: 2m 32s
700:	test: 0.7243845	best: 0.7243845 (700)	total: 4m 25s	remaining: 1m 53s
800:	test: 0.7252912	best: 0.7252912 (800)	total: 4m 59s	remaining: 1m 14s
900:	test: 0.7261244	best: 0.7261244 (900)	total: 5m 34s	remaining: 36.7s
999:	test: 0.7266658	best: 0.7266658 (999)	total: 6m 9s	remaining: 0us

bestTest = 0.7266658342
bestIteration = 999

AUC: 0.726665834216917

Testing: depth=6, lr=0.03, l2_leaf_reg=7
0:	test: 0.5594896	best: 0.5594896

## modeling 14.03 - 2

In [61]:
best_model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    l2_leaf_reg=7,
    loss_function='Logloss',
    eval_metric='AUC',
    early_stopping_rounds=50,
    verbose=100
)

best_model.fit(X_train, y_train, eval_set=(X_val, y_val))

0:	test: 0.5676503	best: 0.5676503 (0)	total: 379ms	remaining: 6m 18s
100:	test: 0.7129855	best: 0.7129855 (100)	total: 44.6s	remaining: 6m 36s
200:	test: 0.7210546	best: 0.7210546 (200)	total: 1m 34s	remaining: 6m 16s
300:	test: 0.7244434	best: 0.7244434 (300)	total: 2m 20s	remaining: 5m 25s
400:	test: 0.7266775	best: 0.7266775 (400)	total: 3m 4s	remaining: 4m 36s
500:	test: 0.7276451	best: 0.7276451 (500)	total: 3m 51s	remaining: 3m 50s
600:	test: 0.7284622	best: 0.7284643 (599)	total: 4m 36s	remaining: 3m 3s
700:	test: 0.7290276	best: 0.7290276 (700)	total: 5m 23s	remaining: 2m 18s
800:	test: 0.7293935	best: 0.7293978 (799)	total: 6m 6s	remaining: 1m 30s
900:	test: 0.7297261	best: 0.7297261 (900)	total: 6m 46s	remaining: 44.7s
999:	test: 0.7300569	best: 0.7300569 (999)	total: 7m 27s	remaining: 0us

bestTest = 0.7300569308
bestIteration = 999



In [62]:
feature_aucs = []

for col in X_train.columns:
    
    auc = roc_auc_score(y_train, X_train[col])
    
    # симметрия AUC
    if auc < 0.5:
        auc = 1 - auc
        
    feature_aucs.append((col, auc))

feature_auc_df = pd.DataFrame(feature_aucs, columns=['feature', 'auc'])

feature_auc_df = feature_auc_df.sort_values('auc', ascending=False)

In [63]:
importances = best_model.get_feature_importance()

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values('importance', ascending=False)

In [64]:
print("TOP 20 features by AUC")
print(feature_auc_df.head(20))

print("\nBOTTOM 20 features by AUC")
print(feature_auc_df.tail(20))

print("TOP 20 important features")
print(importance_df.head(20))

print("\nBOTTOM 20 least important features")
print(importance_df.tail(20))

TOP 20 features by AUC
                   feature       auc
58         enc_paym_2_mean  0.603499
61         enc_paym_3_mean  0.603493
55         enc_paym_1_mean  0.601376
64         enc_paym_4_mean  0.601348
67         enc_paym_5_mean  0.598223
70         enc_paym_6_mean  0.595144
73         enc_paym_7_mean  0.594203
76         enc_paym_8_mean  0.593086
79         enc_paym_9_mean  0.592134
82        enc_paym_10_mean  0.589458
85        enc_paym_11_mean  0.589006
88        enc_paym_12_mean  0.588614
56          enc_paym_1_sum  0.573029
52         enc_paym_0_mean  0.572847
33   is_zero_loans530_mean  0.572250
42       is_zero_util_mean  0.566991
59          enc_paym_2_sum  0.565678
53          enc_paym_0_sum  0.563008
35  is_zero_loans3060_mean  0.561879
34    is_zero_loans530_sum  0.560961

BOTTOM 20 features by AUC
                            feature       auc
78                   enc_paym_9_max  0.509234
81                  enc_paym_10_max  0.507445
18     pre_loans_next_pay_summ_mean

In [65]:
merged = feature_auc_df.merge(importance_df, on='feature')

merged = merged.sort_values('auc', ascending=False)

merged.head(30)

,feature,auc,importance
0,enc_paym_2_mean,0.603499,1.560413
1,enc_paym_3_mean,0.603493,1.125562
2,enc_paym_1_mean,0.601376,1.492875
3,enc_paym_4_mean,0.601348,1.572847
4,enc_paym_5_mean,0.598223,0.441910
5,enc_paym_6_mean,0.595144,0.827449
6,enc_paym_7_mean,0.594203,0.380810
7,enc_paym_8_mean,0.593086,0.843822
8,enc_paym_9_mean,0.592134,1.077759
9,enc_paym_10_mean,0.589458,1.158957


## modeling 15.03

In [14]:
X = data.drop(['flag', 'id'], axis=1)
y = data["flag"]

In [15]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}
print("Class weights:", class_weights_dict)

# stratify=y для сохранения распределения классов
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    train_size=0.7, 
    stratify=y, 
    random_state=42
)

Class weights: {0: np.float64(0.5183929266321947), 1: np.float64(14.092181657616354)}


In [16]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    train_size=0.7, 
    stratify=y, 
    random_state=42
)

## grid

In [17]:
grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05],
    'l2_leaf_reg': [3, 5, 7],
}

best_auc = 0
best_params = None

for depth, lr, l2 in product(grid['depth'], grid['learning_rate'], grid['l2_leaf_reg']):
    print(f"Testing: depth={depth}, lr={lr}, l2_leaf_reg={l2}")
    model = CatBoostClassifier(
        iterations=500,
        depth=depth,
        learning_rate=lr,
        l2_leaf_reg=l2,
        loss_function='Logloss',
        eval_metric='AUC',
        early_stopping_rounds=50,
        verbose=100
    )
    model.fit(X_train, y_train, eval_set=(X_val, y_val))
    
    auc = model.get_best_score()['validation']['AUC']
    print(f"AUC: {auc}\n")
    
    if auc > best_auc:
        best_auc = auc
        best_params = {'depth': depth, 'learning_rate': lr, 'l2_leaf_reg': l2}

print(f"Best AUC: {best_auc} with params {best_params}")

Testing: depth=4, lr=0.01, l2_leaf_reg=3
0:	test: 0.5916312	best: 0.5916312 (0)	total: 414ms	remaining: 3m 26s
100:	test: 0.6805910	best: 0.6805910 (100)	total: 26.4s	remaining: 1m 44s
200:	test: 0.6926423	best: 0.6926423 (200)	total: 53s	remaining: 1m 18s
300:	test: 0.6983356	best: 0.6983391 (299)	total: 1m 19s	remaining: 52.4s
400:	test: 0.7016755	best: 0.7016755 (400)	total: 1m 46s	remaining: 26.2s
499:	test: 0.7040320	best: 0.7040320 (499)	total: 2m 12s	remaining: 0us

bestTest = 0.7040320433
bestIteration = 499

AUC: 0.7040320432822642

Testing: depth=4, lr=0.01, l2_leaf_reg=5
0:	test: 0.5916317	best: 0.5916317 (0)	total: 281ms	remaining: 2m 20s
100:	test: 0.6779538	best: 0.6779538 (100)	total: 26.9s	remaining: 1m 46s
200:	test: 0.6925506	best: 0.6925506 (200)	total: 53.3s	remaining: 1m 19s
300:	test: 0.6979414	best: 0.6979659 (299)	total: 1m 22s	remaining: 54.7s
400:	test: 0.7016435	best: 0.7016435 (400)	total: 1m 49s	remaining: 27.1s
499:	test: 0.7042271	best: 0.7042271 (499)	to

## model

In [18]:
model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=100
)

model.fit(X_train, y_train,
          eval_set=(X_val, y_val))

0:	test: 0.6101524	best: 0.6101524 (0)	total: 365ms	remaining: 6m 4s
100:	test: 0.7025626	best: 0.7025626 (100)	total: 33.4s	remaining: 4m 57s
200:	test: 0.7106542	best: 0.7106542 (200)	total: 1m 8s	remaining: 4m 30s
300:	test: 0.7154525	best: 0.7154525 (300)	total: 1m 42s	remaining: 3m 58s
400:	test: 0.7186004	best: 0.7186004 (400)	total: 2m 17s	remaining: 3m 24s
500:	test: 0.7207682	best: 0.7207682 (500)	total: 2m 51s	remaining: 2m 50s
600:	test: 0.7225753	best: 0.7225753 (600)	total: 3m 25s	remaining: 2m 16s
700:	test: 0.7239624	best: 0.7239624 (700)	total: 4m 6s	remaining: 1m 44s
800:	test: 0.7249017	best: 0.7249017 (800)	total: 4m 45s	remaining: 1m 10s
900:	test: 0.7257138	best: 0.7257138 (900)	total: 5m 22s	remaining: 35.5s
999:	test: 0.7263815	best: 0.7263815 (999)	total: 5m 58s	remaining: 0us

bestTest = 0.7263814622
bestIteration = 999



## modeling 17.03 - 18.03

In [9]:
data = pd.read_csv('full_data3')

In [29]:
X = data.drop(['flag', 'id'], axis=1)
y = data["flag"]

In [30]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}
print("Class weights:", class_weights_dict)

# stratify=y для сохранения распределения классов
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    train_size=0.7, 
    stratify=y, 
    random_state=42
)

Class weights: {0: np.float64(0.5183929266321947), 1: np.float64(14.092181657616354)}


In [13]:
model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    l2_leaf_reg=7,
    loss_function='Logloss',
    eval_metric='AUC',
    class_weights=class_weights_dict,
    early_stopping_rounds=50,
    verbose=100
)

model.fit(X_train, y_train,
          eval_set=(X_val, y_val))

0:	test: 0.6776694	best: 0.6776694 (0)	total: 778ms	remaining: 12m 57s
100:	test: 0.7203855	best: 0.7203855 (100)	total: 56.5s	remaining: 8m 23s
200:	test: 0.7268205	best: 0.7268205 (200)	total: 1m 54s	remaining: 7m 34s
300:	test: 0.7300951	best: 0.7300951 (300)	total: 2m 48s	remaining: 6m 32s
400:	test: 0.7326421	best: 0.7326421 (400)	total: 3m 45s	remaining: 5m 36s
500:	test: 0.7338162	best: 0.7338162 (500)	total: 4m 37s	remaining: 4m 36s
600:	test: 0.7345289	best: 0.7345607 (588)	total: 5m 32s	remaining: 3m 41s
700:	test: 0.7349805	best: 0.7349805 (700)	total: 6m 25s	remaining: 2m 44s
800:	test: 0.7351763	best: 0.7351845 (797)	total: 7m 24s	remaining: 1m 50s
900:	test: 0.7353139	best: 0.7353208 (892)	total: 8m 21s	remaining: 55.1s
999:	test: 0.7355614	best: 0.7355614 (999)	total: 9m 20s	remaining: 0us

bestTest = 0.7355614356
bestIteration = 999



In [19]:
importances = model.get_feature_importance()

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values('importance', ascending=False)

print("TOP 20 important features")
print(importance_df.head(20))

print("\nBOTTOM 20 least important features")
print(importance_df.tail(20))

TOP 20 important features
                             feature  importance
33             is_zero_loans530_mean    5.019822
21        pre_loans_outstanding_mean    3.781118
42                 is_zero_util_mean    3.752217
38                     pre_util_mean    3.572975
23   pre_loans_credit_cost_rate_mean    2.929762
123              paym_bad_count_mean    2.451683
39                      pre_util_max    2.356761
20         pre_loans_outstanding_max    2.249648
18      pre_loans_next_pay_summ_mean    2.081666
35            is_zero_loans3060_mean    2.037159
131        max_utilization_ratio_max    1.976335
16        pre_loans_credit_limit_max    1.964091
49                  pclose_flag_mean    1.963495
31               is_zero_loans5_mean    1.956893
47     enc_loans_credit_type_nunique    1.882100
19       pre_loans_next_pay_summ_max    1.695958
125          paym_nonzero_share_mean    1.686871
8                     pre_pterm_mean    1.682488
17       pre_loans_credit_limit_mean    1.5

## modeling 19.03

In [81]:
X = data.drop(['flag', 'id'], axis=1)
y = data["flag"]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weights_dict = {0: class_weights[0], 1: class_weights[1]}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== FOLD {fold} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=1000,
        depth=8,
        learning_rate=0.05,
        l2_leaf_reg=7,
        loss_function='Logloss',
        eval_metric='AUC',
        class_weights=class_weights_dict,
        early_stopping_rounds=100,
        verbose=200
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = preds

    fold_auc = roc_auc_score(y_val, preds)
    fold_scores.append(fold_auc)

    print(f"FOLD {fold} AUC: {fold_auc:.6f}")

overall_auc = roc_auc_score(y, oof_preds)


===== FOLD 0 =====
0:	test: 0.6783225	best: 0.6783225 (0)	total: 624ms	remaining: 10m 23s


KeyboardInterrupt: 

## feature importance

In [63]:
fi = model.get_feature_importance(prettified=True)

# переименуем для удобства
fi.columns = ['feature', 'importance']

# сортировка по убыванию
fi_sorted = fi.sort_values(by='importance', ascending=False)

# 20 лучших
top_20 = fi_sorted.head(20)
print("=== Top 20 features ===")
print(top_20)

# 20 худших
bottom_20 = fi_sorted.tail(20)
print("\n=== Bottom 20 features ===")
print(bottom_20)

# отдельно проверим новые фичи
new_features = ["util_per_loan", "bad_payment_ratio", "util_stability"]
print("\n=== New features importance ===")
for f in new_features:
    if f in fi_sorted['feature'].values:
        imp = fi_sorted.loc[fi_sorted['feature'] == f, 'importance'].values[0]
        print(f"{f}: {imp:.6f}")
    else:
        print(f"{f}: not found")

=== Top 20 features ===
                               feature  importance
0                is_zero_loans530_mean    4.866229
1           pre_loans_outstanding_mean    3.685992
2                    is_zero_util_mean    3.022558
3      pre_loans_credit_cost_rate_mean    2.904668
4            max_utilization_ratio_max    2.373087
5            pre_loans_outstanding_max    2.303073
6                        util_per_loan    2.257056
7                         pre_util_max    2.232053
8         pre_loans_next_pay_summ_mean    2.073881
9   client_mean_bad_months_per_product    1.996439
10              is_zero_loans3060_mean    1.982344
11                 is_zero_loans5_mean    1.977211
12      client_mean_nonzero_paym_share    1.949083
13          pre_loans_credit_limit_max    1.874081
14                      util_stability    1.840551
15                       pre_util_mean    1.823789
16         pre_loans_credit_limit_mean    1.778194
17                    pclose_flag_mean    1.745844
18     

## ансамбль с lightgbm и xgboost (переобучение)

In [28]:
median_val = data["bad_payment_ratio"].median()
data["bad_payment_ratio"] = data["bad_payment_ratio"].replace([np.inf, -np.inf], median_val)

In [32]:
for col in X.columns:
    if X[col].dtype.kind in 'f':  # только float колонки
        median_val = X[col].replace([np.inf, -np.inf], np.nan).median()
        X[col] = X[col].replace([np.inf, -np.inf], median_val)

In [33]:
cat_model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    l2_leaf_reg=7,
    loss_function='Logloss',
    eval_metric='AUC',
    class_weights=class_weights_dict,
    early_stopping_rounds=100,
    verbose=200
)
cat_model.fit(X, y, use_best_model=True)
cat_preds = cat_model.predict_proba(X)[:, 1]

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)
lgb_model.fit(X, y)
lgb_preds = lgb_model.predict_proba(X)[:, 1]

xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42
)
xgb_model.fit(X, y)
xgb_preds = xgb_model.predict_proba(X)[:, 1]

You should provide test set for use best model. use_best_model parameter has been switched to false value.


0:	total: 689ms	remaining: 11m 28s
200:	total: 2m 49s	remaining: 11m 14s
400:	total: 5m 22s	remaining: 8m 2s
600:	total: 7m 38s	remaining: 5m 4s
800:	total: 9m 55s	remaining: 2m 27s
999:	total: 12m 37s	remaining: 0us
[LightGBM] [Info] Number of positive: 106442, number of negative: 2893558
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.288632 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14820
[LightGBM] [Info] Number of data points in the train set: 3000000, number of used features: 115
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035481 -> initscore=-3.302642
[LightGBM] [Info] Start training from score -3.302642


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:09:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [34]:
ensemble_preds = (cat_preds + lgb_preds + xgb_preds) / 3

# AUC для каждой модели и ансамбля
print("CatBoost AUC:", roc_auc_score(y, cat_preds))
print("LightGBM AUC:", roc_auc_score(y, lgb_preds))
print("XGBoost AUC:", roc_auc_score(y, xgb_preds))
print("Ensemble AUC:", roc_auc_score(y, ensemble_preds))

CatBoost AUC: 0.7773442764733353
LightGBM AUC: 0.7796894363536342
XGBoost AUC: 0.7833135827541727
Ensemble AUC: 0.7823242004945576


## ансамбль с K-fold

In [35]:
cat_params = {
    'iterations': 1000,
    'depth': 8,
    'learning_rate': 0.05,
    'l2_leaf_reg': 7,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'class_weights': class_weights_dict,
    'early_stopping_rounds': 100,
    'verbose': 200
}

lgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': -1,
    'random_state': 42
}

xgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'use_label_encoder': False,
    'eval_metric': 'auc',
    'random_state': 42
}

In [36]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== FOLD {fold} =====")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # CatBoost
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]

    # LightGBM
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(X_train, y_train)
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]

    # XGBoost
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]


===== FOLD 0 =====
0:	test: 0.6778513	best: 0.6778513 (0)	total: 509ms	remaining: 8m 28s
200:	test: 0.7249495	best: 0.7249495 (200)	total: 1m 41s	remaining: 6m 41s
400:	test: 0.7305285	best: 0.7305285 (400)	total: 3m 21s	remaining: 5m
600:	test: 0.7325758	best: 0.7325758 (600)	total: 5m 1s	remaining: 3m 19s
800:	test: 0.7332035	best: 0.7332198 (740)	total: 6m 40s	remaining: 1m 39s
999:	test: 0.7334622	best: 0.7335565 (955)	total: 8m 20s	remaining: 0us

bestTest = 0.7335564962
bestIteration = 955

Shrink model to first 956 iterations.
[LightGBM] [Info] Number of positive: 70962, number of negative: 1929038
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.784445 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14816
[LightGBM] [Info] Number of data points in the train set: 2000000, number of used features: 115
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035481 -> initscore=-3.302632
[LightGBM] [I

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:49:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



===== FOLD 1 =====
0:	test: 0.6846925	best: 0.6846925 (0)	total: 521ms	remaining: 8m 40s
200:	test: 0.7302694	best: 0.7302694 (200)	total: 1m 41s	remaining: 6m 44s
400:	test: 0.7352928	best: 0.7352928 (400)	total: 3m 27s	remaining: 5m 10s
600:	test: 0.7371545	best: 0.7371545 (600)	total: 5m 10s	remaining: 3m 26s
800:	test: 0.7375080	best: 0.7375108 (799)	total: 6m 51s	remaining: 1m 42s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7376187485
bestIteration = 828

Shrink model to first 829 iterations.
[LightGBM] [Info] Number of positive: 70961, number of negative: 1929039
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.775680 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14821
[LightGBM] [Info] Number of data points in the train set: 2000000, number of used features: 115
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035480 -> initscore=-3.302647
[LightGBM] [Info] Start t

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:04:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



===== FOLD 2 =====
0:	test: 0.6774138	best: 0.6774138 (0)	total: 550ms	remaining: 9m 9s
200:	test: 0.7261050	best: 0.7261050 (200)	total: 1m 41s	remaining: 6m 43s
400:	test: 0.7321057	best: 0.7321057 (400)	total: 3m 23s	remaining: 5m 3s
600:	test: 0.7341327	best: 0.7341327 (600)	total: 5m 5s	remaining: 3m 22s
800:	test: 0.7348560	best: 0.7348560 (800)	total: 6m 46s	remaining: 1m 41s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.734912895
bestIteration = 870

Shrink model to first 871 iterations.
[LightGBM] [Info] Number of positive: 70961, number of negative: 1929039
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.819345 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14857
[LightGBM] [Info] Number of data points in the train set: 2000000, number of used features: 115
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035480 -> initscore=-3.302647
[LightGBM] [Info] Start train

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:20:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [37]:
auc_cat = roc_auc_score(y, oof_cat)
auc_lgb = roc_auc_score(y, oof_lgb)
auc_xgb = roc_auc_score(y, oof_xgb)

# простой ансамбль - среднее предсказаний
ensemble_preds = (oof_cat + oof_lgb + oof_xgb) / 3
auc_ensemble = roc_auc_score(y, ensemble_preds)

print(f"\nCatBoost OOF AUC: {auc_cat:.6f}")
print(f"LightGBM OOF AUC: {auc_lgb:.6f}")
print(f"XGBoost OOF AUC: {auc_xgb:.6f}")
print(f"Ensemble OOF AUC: {auc_ensemble:.6f}")


CatBoost OOF AUC: 0.735336
LightGBM OOF AUC: 0.733697
XGBoost OOF AUC: 0.735600
Ensemble OOF AUC: 0.736681


In [40]:
rank_cat = rankdata(oof_cat)
rank_lgb = rankdata(oof_lgb)
rank_xgb = rankdata(oof_xgb)

ensemble_rank = (rank_cat + rank_lgb + rank_xgb) / 3

roc_auc_score(y, ensemble_rank)

0.7371091806315032

In [41]:
stack_X = np.vstack([
    oof_cat,
    oof_lgb,
    oof_xgb
]).T

meta_model = meta_model = CatBoostClassifier(
    iterations=300,
    depth=4,
    learning_rate=0.05,
    verbose=0
)
meta_model.fit(stack_X, y)

stack_preds = meta_model.predict_proba(stack_X)[:, 1]

print("Stacking AUC:", roc_auc_score(y, stack_preds))

Stacking AUC: 0.7377119222006876


## feature importance

**Изучим ценность фич, чтобы двигаться дальше**

In [43]:
feature_auc = []

for col in X.columns:
    try:
        auc = roc_auc_score(y, X[col])
    except:
        auc = 0.5
        
    feature_auc.append((col, auc))

feature_auc_df = pd.DataFrame(feature_auc, columns=['feature', 'auc'])
feature_auc_df = feature_auc_df.sort_values('auc', ascending=False)

feature_auc_df.head(20)

,feature,auc
103,client_mean_bad_months_per_product,0.638736
105,client_mean_nonzero_paym_share,0.638736
107,client_mean_weighted_paym_score,0.614978
106,client_mean_paym_status,0.614978
104,client_mean_severe_bad_months,0.605064
58,enc_paym_3_mean,0.604221
55,enc_paym_2_mean,0.603623
61,enc_paym_4_mean,0.601843
52,enc_paym_1_mean,0.601438
64,enc_paym_5_mean,0.598777


In [44]:
importances = cat_model.get_feature_importance()

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
}).sort_values('importance', ascending=False)

In [51]:
importance_df.head(20)

,feature,importance
30,is_zero_loans530_mean,5.790033
21,pre_loans_outstanding_mean,3.665130
39,is_zero_util_mean,3.623703
23,pre_loans_credit_cost_rate_mean,2.683089
36,pre_util_max,2.546032
20,pre_loans_outstanding_max,2.531198
103,client_mean_bad_months_per_product,2.483325
105,client_mean_nonzero_paym_share,2.193993
28,is_zero_loans5_mean,2.178622
32,is_zero_loans3060_mean,2.174839


In [55]:
sample_idx = np.random.choice(len(X_val), size=100_000, replace=False)

X_val_sample = X_val.iloc[sample_idx]
y_val_sample = y_val.iloc[sample_idx]

perm = permutation_importance(
    cat_model,
    X_val,
    y_val,
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': X.columns,
    'perm_importance': perm.importances_mean
}).sort_values('perm_importance', ascending=False)

In [83]:
full_df = feature_auc_df.merge(importance_df, on='feature')
full_df = full_df.merge(perm_df, on='feature')

full_df = full_df.sort_values('importance', ascending=False)

full_df.head(40)

,feature,auc,importance,perm_importance
115,is_zero_loans530_mean,0.428651,5.790033,-0.007433
47,pre_loans_outstanding_mean,0.534555,3.665130,0.018679
113,is_zero_util_mean,0.433346,3.623703,-0.028709
74,pre_loans_credit_cost_rate_mean,0.495666,2.683089,-0.006602
72,pre_util_max,0.496575,2.546032,-0.005480
49,pre_loans_outstanding_max,0.528906,2.531198,-0.009390
0,client_mean_bad_months_per_product,0.638736,2.483325,-0.014898
1,client_mean_nonzero_paym_share,0.638736,2.193993,-0.011921
102,is_zero_loans5_mean,0.463584,2.178622,-0.002118
112,is_zero_loans3060_mean,0.438289,2.174839,0.000744


In [53]:
corr = X.corr().abs()

high_corr = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)

high_corr.columns = ['feature_1', 'feature_2', 'corr']
high_corr = high_corr[high_corr['corr'] > 0.9]

high_corr.sort_values('corr', ascending=False).head(20)

,feature_1,feature_2,corr
6519,client_mean_paym_status,client_mean_weighted_paym_score,1.000000
6490,client_mean_bad_months_per_product,client_mean_nonzero_paym_share,1.000000
2544,pre_loans5_sum,client_products_count_max,0.999410
2633,pre_loans530_sum,client_products_count_max,0.999095
2460,pre_loans5_sum,pre_loans530_sum,0.998688
6420,enc_paym_21_sum,enc_paym_22_sum,0.997739
6451,enc_paym_22_sum,enc_paym_23_sum,0.997673
6303,enc_paym_18_sum,enc_paym_19_sum,0.996994
6205,enc_paym_16_sum,enc_paym_17_sum,0.996676
6256,enc_paym_17_sum,enc_paym_18_sum,0.996612


# 5. Ensemble Models & Error Analysis

This section focuses on:
- combining models into ensembles
- analyzing model errors
- identifying strengths and weaknesses of different approaches

## modeling 20.03

### базовый ансамбль

In [108]:
cat_params = {
    'iterations': 1000,
    'depth': 8,
    'learning_rate': 0.05,
    'l2_leaf_reg': 7,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'class_weights': class_weights_dict,
    'early_stopping_rounds': 100,
    'verbose': 200
}

lgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': -1,
    'random_state': 42,
    'class_weight': class_weights_dict
}

pos_weight = class_weights_dict[1] / class_weights_dict[0]
xgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'use_label_encoder': False,
    'eval_metric': 'auc',
    'random_state': 42,
    'scale_pos_weight': pos_weight
}

In [109]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== FOLD {fold} =====")
    
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    for col in X_train.columns:
        if X_train[col].dtype.kind in 'f':
            median_val = X_train[col].replace([np.inf, -np.inf], np.nan).median()
            
            X_train[col] = X_train[col].replace([np.inf, -np.inf], median_val)
            X_val[col] = X_val[col].replace([np.inf, -np.inf], median_val)

    # CatBoost
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]

    # LightGBM
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(X_train, y_train)
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]

    # XGBoost
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]


rank_cat = rankdata(oof_cat)
rank_lgb = rankdata(oof_lgb)
rank_xgb = rankdata(oof_xgb)

ensemble_rank = (rank_cat + rank_lgb + rank_xgb) / 3

stack_X = np.vstack([
    oof_cat,
    oof_lgb,
    oof_xgb
]).T

meta_model = CatBoostClassifier(
    iterations=300,
    depth=4,
    learning_rate=0.05,
    verbose=0
)
meta_model.fit(stack_X, y)

stack_preds = meta_model.predict_proba(stack_X)[:, 1]

print("Stacking AUC:", roc_auc_score(y, stack_preds))


===== FOLD 0 =====
0:	test: 0.6783225	best: 0.6783225 (0)	total: 481ms	remaining: 8m
200:	test: 0.7249159	best: 0.7249159 (200)	total: 1m 42s	remaining: 6m 45s
400:	test: 0.7307831	best: 0.7307831 (400)	total: 3m 22s	remaining: 5m 2s
600:	test: 0.7328709	best: 0.7328709 (600)	total: 5m 3s	remaining: 3m 21s
800:	test: 0.7337269	best: 0.7337291 (799)	total: 6m 46s	remaining: 1m 41s
999:	test: 0.7340554	best: 0.7341207 (971)	total: 8m 28s	remaining: 0us

bestTest = 0.7341206983
bestIteration = 971

Shrink model to first 972 iterations.
[LightGBM] [Info] Number of positive: 70962, number of negative: 1929038
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.769538 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15702
[LightGBM] [Info] Number of data points in the train set: 2000000, number of used features: 119
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500002 -> initscore=0.000010
[LightGBM] [Inf

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:43:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



===== FOLD 1 =====
0:	test: 0.6868586	best: 0.6868586 (0)	total: 520ms	remaining: 8m 39s
200:	test: 0.7301273	best: 0.7301273 (200)	total: 1m 43s	remaining: 6m 52s
400:	test: 0.7355632	best: 0.7355632 (400)	total: 3m 28s	remaining: 5m 11s
600:	test: 0.7371962	best: 0.7371962 (600)	total: 5m 11s	remaining: 3m 27s
800:	test: 0.7375765	best: 0.7376100 (793)	total: 6m 55s	remaining: 1m 43s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7376655539
bestIteration = 893

Shrink model to first 894 iterations.
[LightGBM] [Info] Number of positive: 70961, number of negative: 1929039
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.802707 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15707
[LightGBM] [Info] Number of data points in the train set: 2000000, number of used features: 119
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499999 -> initscore=-0.000005
[LightGBM] [Info] Start t

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:00:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



===== FOLD 2 =====
0:	test: 0.6819886	best: 0.6819886 (0)	total: 452ms	remaining: 7m 31s
200:	test: 0.7258046	best: 0.7258046 (200)	total: 1m 43s	remaining: 6m 52s
400:	test: 0.7317767	best: 0.7317767 (400)	total: 3m 28s	remaining: 5m 10s
600:	test: 0.7338715	best: 0.7338717 (598)	total: 5m 11s	remaining: 3m 26s
800:	test: 0.7343777	best: 0.7343841 (799)	total: 6m 54s	remaining: 1m 42s
999:	test: 0.7346971	best: 0.7347317 (977)	total: 8m 36s	remaining: 0us

bestTest = 0.7347316832
bestIteration = 977

Shrink model to first 978 iterations.
[LightGBM] [Info] Number of positive: 70961, number of negative: 1929039
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.819697 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15740
[LightGBM] [Info] Number of data points in the train set: 2000000, number of used features: 119
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499999 -> initscore=-0.000005
[LightGB

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:16:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Stacking AUC: 0.7370911177859396


### lightgbm grid

In [128]:
grid = {
    'num_leaves': [31, 64, 128],
    'min_child_samples': [20, 50],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9]
}

best_auc = 0
best_params = None

for num_leaves, min_child, subsample, colsample in product(
    grid['num_leaves'],
    grid['min_child_samples'],
    grid['subsample'],
    grid['colsample_bytree']
):
    print(f"Testing: num_leaves={num_leaves}, min_child_samples={min_child}, subsample={subsample}, colsample_bytree={colsample}")
    
    model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=num_leaves,
        min_child_samples=min_child,
        subsample=subsample,
        colsample_bytree=colsample,
        random_state=42,
        class_weight=class_weights_dict,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=100)]
    )
    
    preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)
    print(f"AUC: {auc}\n")
    
    if auc > best_auc:
        best_auc = auc
        best_params = {
            'num_leaves': num_leaves,
            'min_child_samples': min_child,
            'subsample': subsample,
            'colsample_bytree': colsample
        }

print(f"Best AUC: {best_auc} with params {best_params}")

Testing: num_leaves=31, min_child_samples=20, subsample=0.7, colsample_bytree=0.7
[LightGBM] [Info] Number of positive: 70961, number of negative: 1929039
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.533224 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15740
[LightGBM] [Info] Number of data points in the train set: 2000000, number of used features: 119
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499999 -> initscore=-0.000005
[LightGBM] [Info] Start training from score -0.000005
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.723825	valid_0's binary_logloss: 0.610751
[200]	valid_0's auc: 0.730575	valid_0's binary_logloss: 0.601841
[300]	valid_0's auc: 0.732112	valid_0's binary_logloss: 0.596665
[400]	valid_0's auc: 0.732996	valid_0's binary_logloss: 0.592269
[500]	valid_0's auc: 0.733421	valid_0's binary_logloss: 0.588193
[600]	valid_0's auc: 0.733618	va

### xgboost grid

In [142]:
pos_weight = class_weights_dict[0] / class_weights_dict[1]

sample_frac = 0.1

idx_0 = np.where(y_train == 0)[0]
idx_1 = np.where(y_train == 1)[0]

n_0 = int(len(idx_0) * sample_frac)
n_1 = int(len(idx_1) * sample_frac)

sample_idx_0 = np.random.choice(idx_0, size=n_0, replace=False)
sample_idx_1 = np.random.choice(idx_1, size=n_1, replace=False)

sample_idx = np.concatenate([sample_idx_0, sample_idx_1])
np.random.shuffle(sample_idx)

X_train_sample = X_train.iloc[sample_idx]
y_train_sample = y_train.iloc[sample_idx]

idx_val_0 = np.where(y_val == 0)[0]
idx_val_1 = np.where(y_val == 1)[0]

n_val_0 = int(len(idx_val_0) * sample_frac)
n_val_1 = int(len(idx_val_1) * sample_frac)

sample_val_idx_0 = np.random.choice(idx_val_0, size=n_val_0, replace=False)
sample_val_idx_1 = np.random.choice(idx_val_1, size=n_val_1, replace=False)

sample_val_idx = np.concatenate([sample_val_idx_0, sample_val_idx_1])
np.random.shuffle(sample_val_idx)

X_val_sample = X_val.iloc[sample_val_idx]
y_val_sample = y_val.iloc[sample_val_idx]

In [143]:
grid = {
    'max_depth': [4, 6, 8],
    'reg_lambda': [1, 5, 10],
    'reg_alpha': [0, 1, 5],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8]
}

param_grid = list(product(
    grid['max_depth'],
    grid['reg_lambda'],
    grid['reg_alpha'],
    grid['subsample'],
    grid['colsample_bytree']
))

best_auc = 0
best_params = None

dtrain = xgb.DMatrix(X_train_sample, label=y_train_sample)
dval = xgb.DMatrix(X_val_sample, label=y_val_sample)

In [144]:
for i, (max_depth, reg_lambda, reg_alpha, subsample, colsample) in enumerate(param_grid, 1):
    print(f"[{i}/{len(param_grid)}] Testing: depth={max_depth}, lambda={reg_lambda}, alpha={reg_alpha}, subsample={subsample}, colsample={colsample}")

    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'max_depth': max_depth,
        'subsample': subsample,
        'colsample_bytree': colsample,
        'reg_lambda': reg_lambda,
        'reg_alpha': reg_alpha,
        'eta': 0.05,
        'scale_pos_weight': pos_weight,
        'seed': 42,
        'nthread': 12,
        'tree_method': 'hist'
    }

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=300,
        evals=[(dval, 'eval')],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    preds = model.predict(dval)
    auc = roc_auc_score(y_val_sample, preds)
    print(f"AUC: {auc:.6f}")

    if auc > best_auc:
        best_auc = auc
        best_params = {
            'max_depth': max_depth,
            'reg_lambda': reg_lambda,
            'reg_alpha': reg_alpha,
            'subsample': subsample,
            'colsample_bytree': colsample
        }
        print(">>> NEW BEST <<<")

print("\nBest AUC:", best_auc)
print("Best params:", best_params)

[1/108] Testing: depth=4, lambda=1, alpha=0, subsample=0.6, colsample=0.6
AUC: 0.720375
>>> NEW BEST <<<
[2/108] Testing: depth=4, lambda=1, alpha=0, subsample=0.6, colsample=0.8
AUC: 0.720291
[3/108] Testing: depth=4, lambda=1, alpha=0, subsample=0.8, colsample=0.6
AUC: 0.721702
>>> NEW BEST <<<
[4/108] Testing: depth=4, lambda=1, alpha=0, subsample=0.8, colsample=0.8
AUC: 0.722051
>>> NEW BEST <<<
[5/108] Testing: depth=4, lambda=1, alpha=1, subsample=0.6, colsample=0.6
AUC: 0.719384
[6/108] Testing: depth=4, lambda=1, alpha=1, subsample=0.6, colsample=0.8
AUC: 0.719952
[7/108] Testing: depth=4, lambda=1, alpha=1, subsample=0.8, colsample=0.6
AUC: 0.720347
[8/108] Testing: depth=4, lambda=1, alpha=1, subsample=0.8, colsample=0.8
AUC: 0.720357
[9/108] Testing: depth=4, lambda=1, alpha=5, subsample=0.6, colsample=0.6
AUC: 0.711882
[10/108] Testing: depth=4, lambda=1, alpha=5, subsample=0.6, colsample=0.8
AUC: 0.712369
[11/108] Testing: depth=4, lambda=1, alpha=5, subsample=0.8, colsamp

### ансамбль с гиперпараметрами

In [173]:
cat_params = {
    'iterations': 1000,
    'depth': 8,
    'learning_rate': 0.05,
    'l2_leaf_reg': 7,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'class_weights': class_weights_dict,
    'early_stopping_rounds': 100,
    'verbose': 200
}

{'num_leaves': 31, 'min_child_samples': 50, 'subsample': 0.7, 'colsample_bytree': 0.7}

lgb_params = {
    'n_estimators': 1000,
    'num_leaves': 31,
    'colsample_bytree': 0.7,
    'min_child_samples': 50,
    'subsample': 0.7,
    'learning_rate': 0.05,
    'max_depth': -1,
    'random_state': 42,
    'class_weight': class_weights_dict
}

pos_weight = class_weights_dict[1] / class_weights_dict[0]
xgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'reg_lambda': 10, 
    'reg_alpha': 0, 
    'subsample': 0.8, 
    'colsample_bytree': 0.8,
    'eval_metric': 'auc',
    'random_state': 42,
    'scale_pos_weight': pos_weight
}

In [174]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== FOLD {fold} =====")
    
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    for col in X_train.columns:
        if X_train[col].dtype.kind in 'f':
            median_val = X_train[col].replace([np.inf, -np.inf], np.nan).median()
            
            X_train[col] = X_train[col].replace([np.inf, -np.inf], median_val)
            X_val[col] = X_val[col].replace([np.inf, -np.inf], median_val)

    # CatBoost
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]

    # LightGBM
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(X_train, y_train)
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]

    # XGBoost
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]


rank_cat = rankdata(oof_cat)
rank_lgb = rankdata(oof_lgb)
rank_xgb = rankdata(oof_xgb)

ensemble_rank = (rank_cat + rank_lgb + rank_xgb) / 3

stack_X = np.vstack([
    oof_cat,
    oof_lgb,
    oof_xgb
]).T

meta_model = CatBoostClassifier(
    iterations=300,
    depth=4,
    learning_rate=0.05,
    verbose=0
)
meta_model.fit(stack_X, y)

stack_preds = meta_model.predict_proba(stack_X)[:, 1]

print("Stacking AUC:", roc_auc_score(y, stack_preds))


===== FOLD 0 =====
0:	test: 0.6808578	best: 0.6808578 (0)	total: 518ms	remaining: 8m 37s
200:	test: 0.7257704	best: 0.7257704 (200)	total: 1m 42s	remaining: 6m 46s
400:	test: 0.7315526	best: 0.7315526 (400)	total: 3m 23s	remaining: 5m 3s
600:	test: 0.7334004	best: 0.7334004 (600)	total: 5m 4s	remaining: 3m 21s
800:	test: 0.7339334	best: 0.7339334 (800)	total: 6m 44s	remaining: 1m 40s
999:	test: 0.7342772	best: 0.7342772 (999)	total: 8m 25s	remaining: 0us

bestTest = 0.7342772286
bestIteration = 999

[LightGBM] [Info] Number of positive: 70962, number of negative: 1929038
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.837383 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17560
[LightGBM] [Info] Number of data points in the train set: 2000000, number of used features: 128
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500002 -> initscore=0.000010
[LightGBM] [Info] Start training from score 0.000

### feature importance

In [146]:
feature_auc = []

for col in X.columns:
    vals = X[col].replace([np.inf, -np.inf], np.nan).fillna(0)
    
    auc = roc_auc_score(y, vals)
    auc_inv = roc_auc_score(y, -vals)
    
    feature_auc.append((col, max(auc, auc_inv)))

feature_auc = sorted(feature_auc, key=lambda x: x[1], reverse=True)

for col, auc in feature_auc[:30]:
    print(f"{col}: {auc:.6f}")

client_mean_bad_months_per_product: 0.638736
client_mean_nonzero_paym_share: 0.638736
bad_months_score: 0.627133
client_mean_weighted_paym_score: 0.614978
client_mean_paym_status: 0.614978
client_mean_severe_bad_months: 0.605064
enc_paym_3_mean: 0.604221
enc_paym_2_mean: 0.603623
enc_paym_4_mean: 0.601843
enc_paym_1_mean: 0.601438
enc_paym_5_mean: 0.598777
enc_paym_6_mean: 0.595305
enc_paym_7_mean: 0.594183
enc_paym_8_mean: 0.593371
recent_paym_mean: 0.592972
enc_paym_9_mean: 0.592704
enc_paym_15_mean: 0.590589
enc_paym_14_mean: 0.590390
enc_paym_16_mean: 0.590182
enc_paym_10_mean: 0.590031
enc_paym_13_mean: 0.589891
enc_paym_11_mean: 0.589282
enc_paym_17_mean: 0.589158
zero_loan_score: 0.588319
enc_paym_12_mean: 0.588315
enc_paym_18_mean: 0.588268
enc_paym_19_mean: 0.587206
enc_paym_20_mean: 0.585861
enc_paym_21_mean: 0.584146
enc_paym_22_mean: 0.582374


In [147]:
for col, auc in feature_auc[30:]:
    print(f"{col}: {auc:.6f}")

enc_paym_23_mean: 0.580759
enc_paym_0_mean: 0.572855
enc_paym_1_sum: 0.572804
is_zero_loans530_mean: 0.571349
util_per_loan: 0.567389
is_zero_util_mean: 0.566654
enc_paym_2_sum: 0.565492
enc_paym_0_sum: 0.562994
is_zero_loans3060_mean: 0.561711
is_zero_loans530_sum: 0.560889
pre_util_mean: 0.558690
enc_paym_3_sum: 0.556406
enc_paym_0_max: 0.555560
enc_paym_1_max: 0.554150
is_zero_loans6090_mean: 0.549696
pre_since_opened_min: 0.548200
is_zero_loans90_mean: 0.547308
enc_paym_4_sum: 0.545887
client_max_weighted_paym_score: 0.545398
is_zero_loans5_sum: 0.543351
enc_paym_2_max: 0.542854
last_paym_status_max: 0.542167
pre_fterm_max: 0.540958
enc_loans_credit_type_nunique: 0.540463
is_zero_maxover2limit_mean: 0.540373
pre_till_fclose_min: 0.539871
pre_pterm_max: 0.537983
pre_till_pclose_min: 0.537340
enc_paym_5_sum: 0.536896
pre_loans_credit_limit_mean: 0.536724
is_zero_loans5_mean: 0.536416
pre_pterm_min: 0.536007
pre_fterm_min: 0.535720
pre_loans530_sum: 0.535686
pre_loans5_sum: 0.535320
u

In [148]:
feature_importance = cat_model.get_feature_importance()
feature_names = X.columns

fi_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importance
}).sort_values(by="importance", ascending=False)

print(fi_df.tail(30))

                             feature  importance
94                   enc_paym_19_sum    0.293732
107  client_mean_weighted_paym_score    0.269750
24                    pre_loans5_sum    0.245819
26                  pre_loans530_max    0.244170
96                   enc_paym_20_sum    0.236175
59                    enc_paym_3_sum    0.226496
90                   enc_paym_17_sum    0.225375
82                   enc_paym_13_sum    0.216299
68                    enc_paym_6_sum    0.214585
84                   enc_paym_14_sum    0.196824
74                    enc_paym_9_sum    0.194060
72                    enc_paym_8_sum    0.188039
109        client_products_count_max    0.187010
98                   enc_paym_21_sum    0.173042
65                    enc_paym_5_sum    0.169605
51                    enc_paym_1_max    0.169473
88                   enc_paym_16_sum    0.168096
76                   enc_paym_10_sum    0.166629
70                    enc_paym_7_sum    0.148177
92                  

In [150]:
print(fi_df.head(30))

                                feature  importance
118                     zero_loan_score    6.175080
21           pre_loans_outstanding_mean    3.714530
39                    is_zero_util_mean    3.128657
23      pre_loans_credit_cost_rate_mean    2.800118
20            pre_loans_outstanding_max    2.262751
36                         pre_util_max    2.135839
111           max_utilization_ratio_max    2.099523
16           pre_loans_credit_limit_max    1.966642
113                       util_per_loan    1.946068
105      client_mean_nonzero_paym_share    1.921919
18         pre_loans_next_pay_summ_mean    1.890411
35                        pre_util_mean    1.883610
114                      util_stability    1.802261
17          pre_loans_credit_limit_mean    1.706610
14                  pre_till_fclose_min    1.692305
44        enc_loans_credit_type_nunique    1.665300
46                     pclose_flag_mean    1.649465
19          pre_loans_next_pay_summ_max    1.649278
110         

#

## modeling 22.03

In [23]:
cat_params = {
    'iterations': 5000,
    'depth': 8,
    'learning_rate': 0.03,
    'l2_leaf_reg': 7,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'class_weights': class_weights_dict,
    'early_stopping_rounds': 200,
    'verbose': 200
}

lgb_params = {
    'n_estimators': 5000,
    'num_leaves': 31,
    'colsample_bytree': 0.7,
    'min_child_samples': 50,
    'subsample': 0.7,
    'learning_rate': 0.03,
    'max_depth': -1,
    'random_state': 42,
    'class_weight': class_weights_dict
}

pos_weight = class_weights_dict[1] / class_weights_dict[0]
xgb_params = {
    'n_estimators': 5000,
    'learning_rate': 0.03,
    'max_depth': 6,
    'reg_lambda': 10,
    'reg_alpha': 0,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'auc',
    'random_state': 42,
    'scale_pos_weight': pos_weight,
    'early_stopping_rounds': 200
}

In [24]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== FOLD {fold} =====")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # === обработка inf ===
    for col in X_train.columns:
        if X_train[col].dtype.kind in 'f':
            median_val = X_train[col].replace([np.inf, -np.inf], np.nan).median()
            X_train[col] = X_train[col].replace([np.inf, -np.inf], median_val)
            X_val[col] = X_val[col].replace([np.inf, -np.inf], median_val)

    # === CatBoost ===
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]

    # === LightGBM (с early stopping) ===
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(200, verbose=False)]
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]

    # === XGBoost (с early stopping) ===
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    

stack_X = np.vstack([
    oof_cat,
    oof_lgb,
    oof_xgb
]).T

stack_oof = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(stack_X, y)):
    print(f"\n===== META FOLD {fold} =====")

    X_train_meta, X_val_meta = stack_X[train_idx], stack_X[val_idx]
    y_train_meta, y_val_meta = y.iloc[train_idx], y.iloc[val_idx]

    meta_model = CatBoostClassifier(
        iterations=1000,
        depth=4,
        learning_rate=0.03,
        loss_function='Logloss',
        eval_metric='AUC',
        verbose=0
    )

    meta_model.fit(
        X_train_meta, y_train_meta,
        eval_set=(X_val_meta, y_val_meta),
        use_best_model=True
    )

    stack_oof[val_idx] = meta_model.predict_proba(X_val_meta)[:, 1]


print("OOF Cat AUC:", roc_auc_score(y, oof_cat))
print("OOF LGB AUC:", roc_auc_score(y, oof_lgb))
print("OOF XGB AUC:", roc_auc_score(y, oof_xgb))
print("STACKING AUC:", roc_auc_score(y, stack_oof))


===== FOLD 0 =====
0:	test: 0.6805462	best: 0.6805462 (0)	total: 759ms	remaining: 1h 3m 16s
200:	test: 0.7230387	best: 0.7230387 (200)	total: 2m 16s	remaining: 54m 10s
400:	test: 0.7295469	best: 0.7295469 (400)	total: 4m 36s	remaining: 52m 50s
600:	test: 0.7335819	best: 0.7335819 (600)	total: 6m 49s	remaining: 49m 56s
800:	test: 0.7358406	best: 0.7358413 (799)	total: 8m 59s	remaining: 47m 8s
1000:	test: 0.7368310	best: 0.7368310 (1000)	total: 11m 14s	remaining: 44m 53s
1200:	test: 0.7373757	best: 0.7373765 (1184)	total: 13m 29s	remaining: 42m 40s
1400:	test: 0.7377066	best: 0.7377066 (1400)	total: 15m 41s	remaining: 40m 17s
1600:	test: 0.7378976	best: 0.7379069 (1572)	total: 17m 55s	remaining: 38m 3s
1800:	test: 0.7379944	best: 0.7380362 (1771)	total: 20m 11s	remaining: 35m 52s
2000:	test: 0.7380065	best: 0.7380512 (1905)	total: 22m 22s	remaining: 33m 32s
2200:	test: 0.7379178	best: 0.7380976 (2081)	total: 24m 20s	remaining: 30m 57s
Stopped by overfitting detector  (200 iterations wai

In [26]:
print(np.corrcoef(oof_cat, oof_xgb))
print(np.corrcoef(oof_cat, oof_lgb))
print(np.corrcoef(oof_xgb, oof_lgb))

[[1.         0.98285169]
 [0.98285169 1.        ]]
[[1.         0.97956222]
 [0.97956222 1.        ]]
[[1.         0.98583012]
 [0.98583012 1.        ]]


## feature importance

In [28]:
cat_model = CatBoostClassifier(**cat_params)
cat_model.fit(X, y, verbose=200)

fi = pd.DataFrame({
    "feature": X.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

0:	total: 909ms	remaining: 1h 15m 43s
200:	total: 2m 38s	remaining: 1h 3m 13s
400:	total: 5m 36s	remaining: 1h 4m 17s
600:	total: 8m 32s	remaining: 1h 2m 30s
800:	total: 11m 12s	remaining: 58m 42s
1000:	total: 13m 41s	remaining: 54m 43s
1200:	total: 16m 3s	remaining: 50m 48s
1400:	total: 18m 25s	remaining: 47m 20s
1600:	total: 20m 55s	remaining: 44m 24s
1800:	total: 23m 16s	remaining: 41m 20s
2000:	total: 25m 39s	remaining: 38m 27s
2200:	total: 28m 6s	remaining: 35m 44s
2400:	total: 30m 36s	remaining: 33m 7s
2600:	total: 33m 25s	remaining: 30m 49s
2800:	total: 36m 16s	remaining: 28m 28s
3000:	total: 39m 5s	remaining: 26m 2s
3200:	total: 41m 48s	remaining: 23m 30s
3400:	total: 44m 29s	remaining: 20m 54s
3600:	total: 47m 25s	remaining: 18m 25s
3800:	total: 49m 57s	remaining: 15m 45s
4000:	total: 52m 12s	remaining: 13m 2s
4200:	total: 54m 28s	remaining: 10m 21s
4400:	total: 56m 44s	remaining: 7m 43s
4600:	total: 59m 6s	remaining: 5m 7s
4800:	total: 1h 1m 22s	remaining: 2m 32s
4999:	total:

In [30]:
fi.head(10)

,feature,importance
123,bad_to_total_ratio,4.132404
23,pre_loans_credit_cost_rate_mean,2.707723
21,pre_loans_outstanding_mean,2.704404
39,is_zero_util_mean,2.509935
30,is_zero_loans530_mean,2.420276
114,client_util_std_max,2.024737
18,pre_loans_next_pay_summ_mean,1.877560
16,pre_loans_credit_limit_max,1.877338
8,pre_pterm_mean,1.852799
35,pre_util_mean,1.832878


In [31]:
feature_auc = []

for col in X.columns:
    vals = X[col].values

    mask = np.isfinite(vals)
    if mask.sum() == 0:
        auc = 0.5
    else:
        try:
            auc = roc_auc_score(y[mask], vals[mask])
        except:
            auc = 0.5

    auc = max(auc, 1 - auc)

    feature_auc.append((col, auc))

df_auc = pd.DataFrame(feature_auc, columns=["feature", "auc"])

df_full = df_auc.merge(fi, on="feature", how="left")

df_full = df_full.sort_values("importance", ascending=False)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df_full)

                                   feature       auc  importance
123                     bad_to_total_ratio  0.657572    4.132404
23         pre_loans_credit_cost_rate_mean  0.504334    2.707723
21              pre_loans_outstanding_mean  0.534555    2.704404
39                       is_zero_util_mean  0.566654    2.509935
30                   is_zero_loans530_mean  0.571349    2.420276
114                    client_util_std_max  0.549568    2.024737
18            pre_loans_next_pay_summ_mean  0.506952    1.877560
16              pre_loans_credit_limit_max  0.505961    1.877338
8                           pre_pterm_mean  0.514604    1.852799
35                           pre_util_mean  0.558690    1.832878
120                          util_per_loan  0.567389    1.817325
11                          pre_fterm_mean  0.518798    1.808169
111              max_utilization_ratio_max  0.546079    1.807023
110             mean_utilization_ratio_max  0.543065    1.728997
2                    pre_

## modeling 23.03

### ансамбль на разных наборах данных

#### списки фич

In [7]:
xgb_features = [
    # --- core payment behavior ---
    "enc_paym_0_mean", "enc_paym_1_mean", "enc_paym_2_mean", "enc_paym_3_mean",
    "enc_paym_4_mean", "enc_paym_5_mean", "enc_paym_6_mean", "enc_paym_7_mean",
    "enc_paym_8_mean", "enc_paym_9_mean", "enc_paym_10_mean", "enc_paym_11_mean",
    "enc_paym_12_mean", "enc_paym_13_mean", "enc_paym_14_mean", "enc_paym_15_mean",
    "enc_paym_16_mean", "enc_paym_17_mean", "enc_paym_18_mean", "enc_paym_19_mean",
    "enc_paym_20_mean", "enc_paym_21_mean", "enc_paym_22_mean", "enc_paym_23_mean",

    # --- sums (динамика накопления) ---
    "enc_paym_0_sum", "enc_paym_1_sum", "enc_paym_2_sum", "enc_paym_3_sum",
    "enc_paym_4_sum", "enc_paym_5_sum", "enc_paym_6_sum", "enc_paym_7_sum",
    "enc_paym_8_sum", "enc_paym_9_sum", "enc_paym_10_sum", "enc_paym_11_sum",
    "enc_paym_19_sum", "enc_paym_20_sum", "enc_paym_21_sum", "enc_paym_22_sum",
    "enc_paym_23_sum",

    # --- max (локальные пики поведения) ---
    "enc_paym_0_max", "enc_paym_1_max",

    # --- aggregated behavioral features ---
    "client_mean_bad_months_per_product",
    "client_mean_nonzero_paym_share",
    "client_mean_weighted_paym_score",
    "client_mean_paym_status",
    "client_mean_severe_bad_months",
    "client_max_weighted_paym_score",
    "client_worst_loan_max",

    # --- advanced behavior ---
    "recent_bad_ratio_max",
    "decayed_paym_score_mean",
    "decayed_paym_score_max",
    "worst_paym_ever",

    # --- dynamics ---
    "paym_std",
    "paym_switches",
    "paym_trend_lr",
    "paym_trend_abs",

    # --- relative behavior ---
    "recent_vs_old",
    "recent_vs_old_early",
    "last_vs_mean"
]

lgb_features = [
    # --- core ratios / meta ---
    "bad_to_total_ratio",
    "util_per_loan",
    "util_stability",

    # --- credit structure ---
    "pre_loans_credit_cost_rate_mean",
    "pre_loans_outstanding_mean",
    "pre_loans_outstanding_max",
    "pre_loans_credit_limit_mean",
    "pre_loans_credit_limit_max",
    "pre_loans_next_pay_summ_mean",
    "pre_loans_next_pay_summ_max",
    "pre_loans_max_overdue_sum_max",

    # --- time-related credit features ---
    "pre_since_opened_min",
    "pre_since_opened_max",
    "pre_since_opened_mean",
    "pre_since_confirmed_min",
    "pre_since_confirmed_max",
    "pre_since_confirmed_mean",

    "pre_pterm_min",
    "pre_pterm_max",
    "pre_pterm_mean",
    "pre_fterm_min",
    "pre_fterm_max",
    "pre_fterm_mean",

    "pre_till_pclose_min",
    "pre_till_pclose_max",
    "pre_till_fclose_min",
    "pre_till_fclose_max",

    # --- utilization ---
    "pre_util_mean",
    "pre_util_max",
    "mean_utilization_ratio_max",
    "max_utilization_ratio_max",
    "client_util_std_max",

    # --- flags ---
    "pclose_flag_mean",
    "fclose_flag_mean",

    # --- categorical diversity ---
    "enc_loans_credit_type_nunique",
    "enc_loans_account_holder_type_nunique",
    "enc_loans_credit_status_nunique",
    "enc_loans_account_cur_nunique",

    # --- loan counts ---
    "pre_loans5_sum",
    "pre_loans530_sum",

    # --- zero indicators ---
    "is_zero_util_mean",
    "is_zero_loans5_mean",
    "is_zero_loans5_sum",
    "is_zero_loans530_mean",
    "is_zero_loans530_sum",
    "is_zero_loans3060_mean",
    "is_zero_loans6090_mean",
    "is_zero_loans90_mean",

    "is_zero_over2limit_mean",
    "is_zero_maxover2limit_mean",

    # --- limits ---
    "pre_over2limit_max",

    # --- client level ---
    "max_credit_share_max",
    "client_products_count_max",

    # --- misc ---
    "last_paym_status_max"
]

#### модель

In [8]:
cat_params = {
    'iterations': 3000,
    'depth': 8,
    'learning_rate': 0.03,
    'l2_leaf_reg': 7,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'class_weights': class_weights_dict,
    'early_stopping_rounds': 200,
    'verbose': 200
}

lgb_params = {
    'n_estimators': 5000,
    'num_leaves': 31,
    'colsample_bytree': 0.7,
    'min_child_samples': 50,
    'subsample': 0.7,
    'learning_rate': 0.03,
    'max_depth': -1,
    'random_state': 42,
    'class_weight': class_weights_dict,
    'verbosity': -1
}

pos_weight = class_weights_dict[1] / class_weights_dict[0]
xgb_params = {
    'n_estimators': 5000,
    'learning_rate': 0.03,
    'max_depth': 6,
    'reg_lambda': 10,
    'reg_alpha': 0,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'auc',
    'random_state': 42,
    'scale_pos_weight': pos_weight,
    'early_stopping_rounds': 200,
    'verbosity': 1
}

In [9]:
X = data.drop(['flag', 'id'], axis=1)
X_lgb = data[lgb_features]
X_xgb = data[xgb_features]

y = data["flag"]

X_np = X.to_numpy(dtype=np.float32)
X_lgb_np = X_lgb.to_numpy(dtype=np.float32)
X_xgb_np = X_xgb.to_numpy(dtype=np.float32)
y_np = y.to_numpy(dtype=np.int8)

print(set(lgb_features) & set(xgb_features))

print(X_lgb.shape)
print(X_xgb.shape)

set()
(3000000, 54)
(3000000, 61)


In [16]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

cat_models_auc = []
lgb_models_auc = []
xgb_models_auc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n========== FOLD {fold} ==========")

    X_train, X_val = X_np[train_idx], X_np[val_idx]
    X_train_lgb, X_val_lgb = X_lgb_np[train_idx], X_lgb_np[val_idx]
    X_train_xgb, X_val_xgb = X_xgb_np[train_idx], X_xgb_np[val_idx]
    y_train, y_val = y_np[train_idx], y_np[val_idx]

    # ===== Обработка inf =====
    for arr_train, arr_val in [(X_train, X_val), (X_train_lgb, X_val_lgb), (X_train_xgb, X_val_xgb)]:
        for i in range(arr_train.shape[1]):
            col_train = arr_train[:, i]
            col_val = arr_val[:, i]

            median_val = np.nanmedian(col_train[np.isfinite(col_train)])

            col_train[~np.isfinite(col_train)] = median_val
            col_val[~np.isfinite(col_val)] = median_val

    # ===== CatBoost =====
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True,
        verbose=200
    )

    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    auc_cat = roc_auc_score(y_val, oof_cat[val_idx])
    best_iter_cat = cat_model.get_best_iteration()

    print(f"[CatBoost]  AUC: {auc_cat:.6f} | best_iter: {best_iter_cat}")

    cat_models_auc.append((auc_cat, cat_model))

    # ===== LightGBM =====
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_train_lgb, y_train,
        eval_set=[(X_val_lgb, y_val)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(200)
        ]
    )

    oof_lgb[val_idx] = lgb_model.predict_proba(X_val_lgb)[:, 1]
    auc_lgb = roc_auc_score(y_val, oof_lgb[val_idx])
    best_iter_lgb = lgb_model.best_iteration_

    print(f"[LightGBM] AUC: {auc_lgb:.6f} | best_iter: {best_iter_lgb}")

    lgb_models_auc.append((auc_lgb, lgb_model))

    # ===== XGBoost =====
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train_xgb, y_train,
        eval_set=[(X_val_xgb, y_val)],
        verbose=200
    )

    oof_xgb[val_idx] = xgb_model.predict_proba(X_val_xgb)[:, 1]
    auc_xgb = roc_auc_score(y_val, oof_xgb[val_idx])
    best_iter_xgb = xgb_model.best_iteration

    print(f"[XGBoost]  AUC: {auc_xgb:.6f} | best_iter: {best_iter_xgb}")

    xgb_models_auc.append((auc_xgb, xgb_model))


print("\n========== OOF RESULTS ==========")
print("CatBoost AUC:", roc_auc_score(y, oof_cat))
print("LightGBM AUC:", roc_auc_score(y, oof_lgb))
print("XGBoost AUC:", roc_auc_score(y, oof_xgb))


# ===== STACKING =====
stack_X = np.vstack([oof_cat, oof_lgb, oof_xgb]).T
stack_oof = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(stack_X, y)):
    print(f"\n===== META FOLD {fold} =====")

    X_train_meta, X_val_meta = stack_X[train_idx], stack_X[val_idx]
    y_train_meta, y_val_meta = y.iloc[train_idx], y.iloc[val_idx]

    meta_model = CatBoostClassifier(
        iterations=1000,
        depth=4,
        learning_rate=0.03,
        loss_function='Logloss',
        eval_metric='AUC',
        verbose=200
    )

    meta_model.fit(
        X_train_meta, y_train_meta,
        eval_set=(X_val_meta, y_val_meta),
        use_best_model=True
    )

    stack_oof[val_idx] = meta_model.predict_proba(X_val_meta)[:, 1]
    auc_meta = roc_auc_score(y_val_meta, stack_oof[val_idx])

    print(f"[META] AUC: {auc_meta:.6f}")


print("\n========== FINAL ==========")
print("STACKING AUC:", roc_auc_score(y, stack_oof))


========== FOLD 0 ==========
0:	test: 0.6834537	best: 0.6834537 (0)	total: 714ms	remaining: 35m 40s
200:	test: 0.7231019	best: 0.7231019 (200)	total: 1m 49s	remaining: 25m 20s
400:	test: 0.7297244	best: 0.7297244 (400)	total: 3m 38s	remaining: 23m 36s
600:	test: 0.7336456	best: 0.7336456 (600)	total: 5m 28s	remaining: 21m 51s
800:	test: 0.7356999	best: 0.7356999 (800)	total: 7m 19s	remaining: 20m 6s
1000:	test: 0.7366919	best: 0.7366919 (1000)	total: 9m 9s	remaining: 18m 17s
1200:	test: 0.7373700	best: 0.7373700 (1200)	total: 10m 59s	remaining: 16m 27s
1400:	test: 0.7378742	best: 0.7378786 (1398)	total: 12m 49s	remaining: 14m 38s
1600:	test: 0.7380361	best: 0.7380653 (1587)	total: 14m 39s	remaining: 12m 48s
1800:	test: 0.7379986	best: 0.7380893 (1726)	total: 16m 30s	remaining: 10m 59s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7380893298
bestIteration = 1726

Shrink model to first 1727 iterations.
[CatBoost]  AUC: 0.738089 | best_iter: 1726
Training until val

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] AUC: 0.735313 | best_iter: 2043
[0]	validation_0-auc:0.65911
[200]	validation_0-auc:0.67886
[400]	validation_0-auc:0.67936
[600]	validation_0-auc:0.67926
[656]	validation_0-auc:0.67908
[XGBoost]  AUC: 0.679438 | best_iter: 456

========== FOLD 1 ==========
0:	test: 0.6810752	best: 0.6810752 (0)	total: 578ms	remaining: 28m 53s
200:	test: 0.7215847	best: 0.7215847 (200)	total: 1m 53s	remaining: 26m 15s
400:	test: 0.7281368	best: 0.7281368 (400)	total: 3m 46s	remaining: 24m 26s
600:	test: 0.7319052	best: 0.7319052 (600)	total: 5m 39s	remaining: 22m 33s
800:	test: 0.7339128	best: 0.7339198 (799)	total: 7m 31s	remaining: 20m 39s
1000:	test: 0.7349123	best: 0.7349123 (1000)	total: 9m 21s	remaining: 18m 41s
1200:	test: 0.7354124	best: 0.7354348 (1166)	total: 11m 13s	remaining: 16m 48s
1400:	test: 0.7357615	best: 0.7357623 (1399)	total: 13m 2s	remaining: 14m 53s
1600:	test: 0.7360310	best: 0.7360310 (1600)	total: 14m 53s	remaining: 13m 1s
1800:	test: 0.7361526	best: 0.7361560 (1787)

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] AUC: 0.732554 | best_iter: 1749
[0]	validation_0-auc:0.65838
[200]	validation_0-auc:0.67890
[400]	validation_0-auc:0.67966
[600]	validation_0-auc:0.67971
[738]	validation_0-auc:0.67951
[XGBoost]  AUC: 0.679792 | best_iter: 538

========== FOLD 2 ==========
0:	test: 0.6888447	best: 0.6888447 (0)	total: 560ms	remaining: 28m
200:	test: 0.7282154	best: 0.7282154 (200)	total: 1m 54s	remaining: 26m 35s
400:	test: 0.7342952	best: 0.7342952 (400)	total: 3m 53s	remaining: 25m 13s
600:	test: 0.7377291	best: 0.7377291 (600)	total: 5m 50s	remaining: 23m 19s
800:	test: 0.7395938	best: 0.7395938 (800)	total: 7m 46s	remaining: 21m 21s
1000:	test: 0.7404211	best: 0.7404211 (1000)	total: 9m 45s	remaining: 19m 28s
1200:	test: 0.7408593	best: 0.7408599 (1199)	total: 11m 37s	remaining: 17m 25s
1400:	test: 0.7412992	best: 0.7413009 (1398)	total: 13m 32s	remaining: 15m 27s
1600:	test: 0.7413864	best: 0.7413959 (1537)	total: 15m 25s	remaining: 13m 28s
1800:	test: 0.7414234	best: 0.7414234 (1800)	t

...

[LightGBM] AUC: 0.738590 | best_iter: 1743
[0]	validation_0-auc:0.66322
[200]	validation_0-auc:0.68189
[400]	validation_0-auc:0.68248
[600]	validation_0-auc:0.68238
[613]	validation_0-auc:0.68233
[XGBoost]  AUC: 0.682544 | best_iter: 413

========== FOLD 3 ==========
0:	test: 0.6814706	best: 0.6814706 (0)	total: 566ms	remaining: 28m 16s
200:	test: 0.7226057	best: 0.7226057 (200)	total: 1m 52s	remaining: 26m 2s
400:	test: 0.7293950	best: 0.7293950 (400)	total: 3m 44s	remaining: 24m 15s
600:	test: 0.7334452	best: 0.7334452 (600)	total: 5m 38s	remaining: 22m 31s
800:	test: 0.7356744	best: 0.7356744 (800)	total: 7m 30s	remaining: 20m 37s
1000:	test: 0.7368393	best: 0.7368393 (1000)	total: 9m 21s	remaining: 18m 40s
1200:	test: 0.7374594	best: 0.7374594 (1200)	total: 11m 13s	remaining: 16m 48s
1400:	test: 0.7377014	best: 0.7377014 (1400)	total: 13m 1s	remaining: 14m 51s
1600:	test: 0.7378732	best: 0.7378841 (1551)	total: 14m 53s	remaining: 13m
1800:	test: 0.7379071	best: 0.7379181 (1620)	tot

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] AUC: 0.734857 | best_iter: 1401
[0]	validation_0-auc:0.65701
[200]	validation_0-auc:0.67700
[400]	validation_0-auc:0.67753
[600]	validation_0-auc:0.67727
[608]	validation_0-auc:0.67730
[XGBoost]  AUC: 0.677578 | best_iter: 408

========== FOLD 4 ==========
0:	test: 0.6828252	best: 0.6828252 (0)	total: 536ms	remaining: 26m 46s
200:	test: 0.7233731	best: 0.7233731 (200)	total: 1m 52s	remaining: 26m 4s
400:	test: 0.7297631	best: 0.7297631 (400)	total: 3m 45s	remaining: 24m 22s
600:	test: 0.7337004	best: 0.7337004 (600)	total: 5m 39s	remaining: 22m 35s
800:	test: 0.7357986	best: 0.7357986 (800)	total: 7m 31s	remaining: 20m 38s
1000:	test: 0.7369424	best: 0.7369424 (1000)	total: 9m 22s	remaining: 18m 43s
1200:	test: 0.7376149	best: 0.7376149 (1200)	total: 11m 14s	remaining: 16m 50s
1400:	test: 0.7380087	best: 0.7380087 (1400)	total: 13m 4s	remaining: 14m 55s
1600:	test: 0.7381566	best: 0.7381613 (1599)	total: 14m 57s	remaining: 13m 3s
1800:	test: 0.7382060	best: 0.7382717 (1709)	

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] AUC: 0.736200 | best_iter: 2495
[0]	validation_0-auc:0.65780
[200]	validation_0-auc:0.67922
[400]	validation_0-auc:0.68000
[600]	validation_0-auc:0.67991
[672]	validation_0-auc:0.67979
[XGBoost]  AUC: 0.680077 | best_iter: 472

========== OOF RESULTS ==========
CatBoost AUC: 0.7383799287438716
LightGBM AUC: 0.735461521776563
XGBoost AUC: 0.6798668074956296

===== META FOLD 0 =====
0:	test: 0.7185058	best: 0.7185058 (0)	total: 196ms	remaining: 3m 16s
200:	test: 0.7386652	best: 0.7386652 (200)	total: 37.4s	remaining: 2m 28s
400:	test: 0.7387962	best: 0.7387979 (378)	total: 1m 14s	remaining: 1m 50s
600:	test: 0.7388186	best: 0.7388254 (543)	total: 1m 50s	remaining: 1m 13s
800:	test: 0.7388164	best: 0.7388260 (668)	total: 2m 28s	remaining: 36.9s
999:	test: 0.7387874	best: 0.7388260 (668)	total: 3m 5s	remaining: 0us

bestTest = 0.7388260252
bestIteration = 668

Shrink model to first 669 iterations.
[META] AUC: 0.738826

===== META FOLD 1 =====
0:	test: 0.7169945	best: 0.7169945 (

In [19]:
models_dir = "models"
os.makedirs(models_dir, exist_ok=True)

# ===== Сохраняем CatBoost модели =====
for i, (auc, model) in enumerate(cat_models_auc):
    filename = os.path.join(models_dir, f"catboost_fold{i}.cbm")
    model.save_model(filename)
    print(f"Saved CatBoost fold {i} -> {filename}")

# ===== Сохраняем LightGBM модели =====
for i, (auc, model) in enumerate(lgb_models_auc):
    filename = os.path.join(models_dir, f"lgbm_fold{i}.pkl")
    joblib.dump(model, filename)
    print(f"Saved LightGBM fold {i} -> {filename}")

# ===== Сохраняем XGBoost модели =====
for i, (auc, model) in enumerate(xgb_models_auc):
    filename = os.path.join(models_dir, f"xgb_fold{i}.json")
    model.save_model(filename)
    print(f"Saved XGBoost fold {i} -> {filename}")

# ===== Сохраняем метамодель =====
meta_filename = os.path.join(models_dir, "meta_model.cbm")
meta_model.save_model(meta_filename)
print(f"Saved meta model -> {meta_filename}")

Saved CatBoost fold 0 -> models\catboost_fold0.cbm
Saved CatBoost fold 1 -> models\catboost_fold1.cbm
Saved CatBoost fold 2 -> models\catboost_fold2.cbm
Saved CatBoost fold 3 -> models\catboost_fold3.cbm
Saved CatBoost fold 4 -> models\catboost_fold4.cbm
Saved LightGBM fold 0 -> models\lgbm_fold0.pkl
Saved LightGBM fold 1 -> models\lgbm_fold1.pkl
Saved LightGBM fold 2 -> models\lgbm_fold2.pkl
Saved LightGBM fold 3 -> models\lgbm_fold3.pkl
Saved LightGBM fold 4 -> models\lgbm_fold4.pkl
Saved XGBoost fold 0 -> models\xgb_fold0.json
Saved XGBoost fold 1 -> models\xgb_fold1.json
Saved XGBoost fold 2 -> models\xgb_fold2.json
Saved XGBoost fold 3 -> models\xgb_fold3.json
Saved XGBoost fold 4 -> models\xgb_fold4.json
Saved meta model -> models\meta_model.cbm


## modeling 03.04

### загружаем модели, восстанавливаем данные предсказаний

In [4]:
models_dir = "models"

cat_models = []
lgb_models = []
xgb_models = []

# CatBoost
for i in range(5):
    model = CatBoostClassifier()
    model.load_model(os.path.join(models_dir, f"catboost_fold{i}.cbm"))
    cat_models.append(model)

# LightGBM
for i in range(5):
    model = joblib.load(os.path.join(models_dir, f"lgbm_fold{i}.pkl"))
    lgb_models.append(model)

# XGBoost
for i in range(5):
    model = xgb.XGBClassifier()
    model.load_model(os.path.join(models_dir, f"xgb_fold{i}.json"))
    xgb_models.append(model)

In [10]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

for fold, (_, val_idx) in enumerate(skf.split(X, y)):
    print(f"Restoring fold {fold}")
    
    # CatBoost
    oof_cat[val_idx] = cat_models[fold].predict_proba(X_np[val_idx])[:, 1]
    
    # LightGBM
    oof_lgb[val_idx] = lgb_models[fold].predict_proba(X_lgb_np[val_idx])[:, 1]
    
    # XGBoost
    oof_xgb[val_idx] = xgb_models[fold].predict_proba(X_xgb_np[val_idx])[:, 1]

Restoring fold 0


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Restoring fold 1


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Restoring fold 2


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Restoring fold 3


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Restoring fold 4


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [11]:
print("CatBoost OOF AUC:", roc_auc_score(y, oof_cat))
print("LightGBM OOF AUC:", roc_auc_score(y, oof_lgb))
print("XGBoost OOF AUC:", roc_auc_score(y, oof_xgb))

CatBoost OOF AUC: 0.7383799287438716
LightGBM OOF AUC: 0.735461521776563
XGBoost OOF AUC: 0.6798668074956296


### выясняем ошибки

In [12]:
df = data.copy()

df["target"] = y

df["oof_cat"] = oof_cat
df["oof_lgb"] = oof_lgb
df["oof_xgb"] = oof_xgb

df["oof_mean"] = (oof_cat + oof_lgb) / 2

df["error"] = abs(df["target"] - df["oof_mean"])

df["residual"] = df["target"] - df["oof_mean"]

df["confidence"] = abs(df["oof_mean"] - 0.5)

df["model_disagreement"] = (
    abs(df["oof_cat"] - df["oof_lgb"]) +
    abs(df["oof_cat"] - df["oof_xgb"]) +
    abs(df["oof_lgb"] - df["oof_xgb"])
)

In [13]:
df["bin"] = pd.qcut(df["oof_mean"], 10, duplicates="drop")

analysis = df.groupby("bin").agg({
    "target": "mean",
    "oof_mean": "mean",
    "error": "mean",
    "residual": "mean",
    "confidence": "mean",
    "model_disagreement": "mean"
}).reset_index()

print(analysis)

               bin    target  oof_mean     error  residual  confidence  \
0  (0.0093, 0.157]  0.004787  0.114493  0.118107 -0.109706    0.385507   
1    (0.157, 0.22]  0.008270  0.189605  0.194694 -0.181335    0.310395   
2    (0.22, 0.278]  0.012340  0.249391  0.255534 -0.237051    0.250609   
3   (0.278, 0.336]  0.017007  0.306890  0.313408 -0.289883    0.193110   
4   (0.336, 0.394]  0.022323  0.364715  0.370686 -0.342391    0.135285   
5   (0.394, 0.454]  0.028687  0.423876  0.428158 -0.395189    0.076124   
6   (0.454, 0.517]  0.037140  0.485108  0.486117 -0.447968    0.019314   
7   (0.517, 0.586]  0.047477  0.550625  0.545683 -0.503148    0.050625   
8    (0.586, 0.67]  0.064287  0.625740  0.609257 -0.561454    0.125740   
9    (0.67, 0.977]  0.112490  0.740428  0.682700 -0.627938    0.240428   

   model_disagreement  
0            0.377319  
1            0.323880  
2            0.281809  
3            0.248528  
4            0.226809  
5            0.218339  
6            0.22

C:\Users\User\AppData\Local\Temp\ipykernel_19640\3548515041.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis = df.groupby("bin").agg({


In [14]:
calibrator = LogisticRegression(max_iter=1000)
calibrator.fit(df["oof_mean"].values.reshape(-1, 1), df["target"])

df["oof_calibrated"] = calibrator.predict_proba(
    df["oof_mean"].values.reshape(-1, 1)
)[:, 1]

In [15]:
df["bin_calibrated"] = pd.qcut(df["oof_calibrated"], 10, duplicates="drop")

analysis_calibrated = df.groupby("bin_calibrated").agg({
    "target": "mean",
    "oof_calibrated": "mean",
    "error": "mean",
    "residual": "mean",
    "confidence": "mean"
}).reset_index()

print(analysis_calibrated)

       bin_calibrated    target  oof_calibrated     error  residual  \
0  (0.00288, 0.00777]  0.004787        0.006407  0.118107 -0.109706   
1   (0.00777, 0.0104]  0.008270        0.009063  0.194694 -0.181335   
2    (0.0104, 0.0137]  0.012340        0.011987  0.255534 -0.237051   
3    (0.0137, 0.0179]  0.017007        0.015676  0.313408 -0.289883   
4    (0.0179, 0.0234]  0.022323        0.020512  0.370686 -0.342391   
5    (0.0234, 0.0309]  0.028687        0.026966  0.428158 -0.395189   
6    (0.0309, 0.0411]  0.037140        0.035718  0.486117 -0.447968   
7    (0.0411, 0.0561]  0.047477        0.048106  0.545683 -0.503148   
8    (0.0561, 0.0813]  0.064287        0.067373  0.609257 -0.561454   
9     (0.0813, 0.274]  0.112490        0.112480  0.682700 -0.627938   

   confidence  
0    0.385507  
1    0.310395  
2    0.250609  
3    0.193110  
4    0.135285  
5    0.076124  
6    0.019314  
7    0.050625  
8    0.125740  
9    0.240428  


C:\Users\User\AppData\Local\Temp\ipykernel_19640\3605508851.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis_calibrated = df.groupby("bin_calibrated").agg({


In [16]:
df.sort_values("error", ascending=False).head(50)

,id,pre_since_opened_min,pre_since_opened_max,pre_since_opened_mean,pre_since_confirmed_min,pre_since_confirmed_max,pre_since_confirmed_mean,pre_pterm_min,pre_pterm_max,pre_pterm_mean,...,oof_lgb,oof_xgb,oof_mean,error,residual,confidence,model_disagreement,bin,oof_calibrated,bin_calibrated
824981,2824981,8,16,12.000,1,17,7.727,1,13,7.637,...,0.015737,0.206148,0.016884,0.983116,0.983116,0.483116,0.380824,"(0.0093, 0.157]",0.004007,"(0.00288, 0.00777]"
1948814,1448814,12,19,12.710,2,16,9.190,3,14,3.975,...,0.976090,0.826998,0.976731,0.976731,-0.976731,0.476731,0.300748,"(0.67, 0.977]",0.274492,"(0.0813, 0.274]"
2103828,1603828,7,18,13.664,6,16,10.664,0,12,5.168,...,0.021151,0.158201,0.025598,0.974402,0.974402,0.474402,0.274100,"(0.0093, 0.157]",0.004175,"(0.00288, 0.00777]"
2523230,2023230,4,18,11.000,0,17,8.500,0,17,7.750,...,0.025324,0.140955,0.029331,0.970669,0.970669,0.470669,0.231261,"(0.0093, 0.157]",0.004250,"(0.00288, 0.00777]"
2910075,2410075,3,19,12.930,0,16,6.105,3,14,4.320,...,0.969878,0.779866,0.970597,0.970597,-0.970597,0.470597,0.382900,"(0.67, 0.977]",0.268747,"(0.0813, 0.274]"
1991164,1491164,3,15,8.700,7,12,9.300,1,14,3.000,...,0.019584,0.219688,0.029754,0.970246,0.970246,0.470246,0.400207,"(0.0093, 0.157]",0.004258,"(0.00288, 0.00777]"
2936945,2436945,2,19,12.880,0,16,7.560,0,14,4.500,...,0.963157,0.808360,0.968720,0.968720,-0.968720,0.468720,0.331845,"(0.67, 0.977]",0.267005,"(0.0813, 0.274]"
1320099,820099,4,17,10.500,9,9,9.000,4,4,4.000,...,0.022629,0.274379,0.032436,0.967564,0.967564,0.467564,0.503499,"(0.0093, 0.157]",0.004312,"(0.00288, 0.00777]"
1413586,913586,2,19,12.860,2,16,9.690,0,13,4.277,...,0.957016,0.780019,0.966591,0.966591,-0.966591,0.466591,0.392291,"(0.67, 0.977]",0.265037,"(0.0813, 0.274]"
919399,2919399,3,16,6.500,3,9,5.500,4,16,10.500,...,0.046133,0.393994,0.037676,0.962324,0.962324,0.462324,0.729547,"(0.0093, 0.157]",0.004420,"(0.00288, 0.00777]"


### ещё больше метрик

In [17]:
threshold = 0.5

df["error_type"] = "TN"

df.loc[(df["target"] == 1) & (df["oof_mean"] >= threshold), "error_type"] = "TP"
df.loc[(df["target"] == 1) & (df["oof_mean"] < threshold), "error_type"] = "FN"
df.loc[(df["target"] == 0) & (df["oof_mean"] >= threshold), "error_type"] = "FP"

In [18]:
error_stats = df.groupby("error_type").agg({
    "target": "mean",
    "oof_mean": "mean",
    "error": "mean",
    "residual": "mean",
    "confidence": "mean",
    "model_disagreement": "mean"
}).sort_values("error", ascending=False)

print("\n=== ERROR STATS ===")
print(error_stats)


=== ERROR STATS ===
            target  oof_mean     error  residual  confidence  \
error_type                                                     
FN             1.0  0.363166  0.636834  0.636834    0.136834   
FP             0.0  0.625300  0.625300 -0.625300    0.125300   
TP             1.0  0.668867  0.331133  0.331133    0.168867   
TN             0.0  0.295743  0.295743 -0.295743    0.204257   

            model_disagreement  
error_type                      
FN                    0.244924  
FP                    0.237364  
TP                    0.241511  
TN                    0.273427  


In [19]:
error_counts = df["error_type"].value_counts(normalize=True)

print("\n=== ERROR DISTRIBUTION ===")
print(error_counts)


=== ERROR DISTRIBUTION ===
error_type
TN    0.661779
FP    0.302741
TP    0.023491
FN    0.011989
Name: proportion, dtype: float64


In [24]:
# Выбираем только числовые признаки (исключаем служебные)
exclude_cols = [
    "target", "oof_cat", "oof_lgb", "oof_xgb",
    "oof_mean", "error", "residual", "confidence",
    "model_disagreement", "error_type",
    "bin", "bin_calibrated"
]

feature_cols = [col for col in df.columns if col not in exclude_cols and df[col].dtype != "object"]

# Средние значения признаков по типам ошибок
feature_analysis = df.groupby("error_type")[feature_cols].mean()

print("\n=== FEATURE MEAN BY ERROR TYPE ===")
print(feature_analysis)


=== FEATURE MEAN BY ERROR TYPE ===
                      id  pre_since_opened_min  pre_since_opened_max  \
error_type                                                             
FN          1.586132e+06              2.458074             15.867299   
FP          1.478092e+06              3.742802             15.855054   
TN          1.510030e+06              2.418896             15.903067   
TP          1.455790e+06              4.190240             15.722465   

            pre_since_opened_mean  pre_since_confirmed_min  \
error_type                                                   
FN                       9.093937                 2.912561   
FP                       9.657073                 3.332025   
TN                       9.164279                 2.919392   
TP                       9.846623                 3.501376   

            pre_since_confirmed_max  pre_since_confirmed_mean  pre_pterm_min  \
error_type                                                                    

In [25]:
if "FN" in df["error_type"].values and "TP" in df["error_type"].values:
    fn_mean = df[df["error_type"] == "FN"][feature_cols].mean()
    tp_mean = df[df["error_type"] == "TP"][feature_cols].mean()
    
    diff_fn_tp = (fn_mean - tp_mean).abs().sort_values(ascending=False)

    print("\n=== TOP FEATURES: FN vs TP (model misses positives) ===")
    print(diff_fn_tp.head(20))


=== TOP FEATURES: FN vs TP (model misses positives) ===
max_utilization_ratio_max             580287.997579
client_util_std_max                   204006.460197
id                                    130341.266766
mean_utilization_ratio_max             94118.276886
pre_loans530_sum                          25.663539
client_mean_weighted_paym_score           10.657897
pre_loans5_sum                             9.567960
decayed_paym_score_mean                    4.583526
client_mean_bad_months_per_product         3.720217
enc_paym_2_sum                             3.482076
client_mean_severe_bad_months              3.419420
enc_paym_3_sum                             3.401044
client_max_weighted_paym_score             3.219564
enc_paym_4_sum                             3.192154
enc_paym_20_sum                            3.131614
enc_paym_1_sum                             2.938739
enc_paym_5_sum                             2.877022
enc_paym_6_sum                             2.413448
decayed

In [22]:
hypothesis_check = df.groupby("error_type")[[
    "model_disagreement",
    "confidence",
    "oof_mean"
]].mean()

print("\n=== HYPOTHESIS CHECK ===")
print(hypothesis_check)


=== HYPOTHESIS CHECK ===
            model_disagreement  confidence  oof_mean
error_type                                          
FN                    0.244924    0.136834  0.363166
FP                    0.237364    0.125300  0.625300
TN                    0.273427    0.204257  0.295743
TP                    0.241511    0.168867  0.668867


In [28]:
abs(df.groupby("error_type").mean(numeric_only=True).loc["FP"] -
    df.groupby("error_type").mean(numeric_only=True).loc["TN"]).sort_values(ascending=False).head(20)

max_utilization_ratio_max             564369.320434
client_util_std_max                   215543.825239
mean_utilization_ratio_max            109653.219973
id                                     31938.337638
pre_loans530_sum                          16.792884
client_mean_weighted_paym_score           10.147324
pre_loans5_sum                             6.113615
decayed_paym_score_mean                    3.975303
client_mean_bad_months_per_product         3.633835
client_max_weighted_paym_score             3.287662
client_mean_severe_bad_months              3.200703
enc_paym_3_sum                             2.998331
enc_paym_2_sum                             2.965600
enc_paym_4_sum                             2.893129
enc_paym_5_sum                             2.689523
enc_paym_1_sum                             2.498426
enc_paym_6_sum                             2.391644
enc_paym_7_sum                             2.184223
decayed_paym_score_max                     1.987949
enc_paym_8_s

## modeling 6.04-7.04

In [13]:
cat_params = {
    'iterations': 3000,
    'depth': 8,
    'learning_rate': 0.03,
    'l2_leaf_reg': 7,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'class_weights': class_weights_dict,
    'early_stopping_rounds': 200,
    'verbose': 200
}

In [14]:
X = data.drop(['flag', 'id'], axis=1)
y = data["flag"]

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== FOLD {fold} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)

    model.fit(X_train, y_train, eval_set=(X_val, y_val))

    preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = preds

    fold_auc = roc_auc_score(y_val, preds)
    scores.append(fold_auc)

    print(f"Fold AUC: {fold_auc:.5f}")

print("\nMean AUC:", np.mean(scores))
print("OOF AUC:", roc_auc_score(y, oof_preds))


===== FOLD 0 =====
0:	test: 0.6834223	best: 0.6834223 (0)	total: 705ms	remaining: 35m 14s
200:	test: 0.7237164	best: 0.7237164 (200)	total: 1m 46s	remaining: 24m 41s
400:	test: 0.7303229	best: 0.7303229 (400)	total: 3m 35s	remaining: 23m 16s
600:	test: 0.7341564	best: 0.7341564 (600)	total: 5m 32s	remaining: 22m 7s
800:	test: 0.7362640	best: 0.7362640 (800)	total: 7m 38s	remaining: 20m 59s
1000:	test: 0.7371618	best: 0.7371618 (1000)	total: 9m 39s	remaining: 19m 17s
1200:	test: 0.7378215	best: 0.7378301 (1192)	total: 11m 25s	remaining: 17m 7s
1400:	test: 0.7380649	best: 0.7380694 (1396)	total: 13m 16s	remaining: 15m 9s
1600:	test: 0.7381901	best: 0.7381932 (1565)	total: 15m 14s	remaining: 13m 19s
1800:	test: 0.7381919	best: 0.7382488 (1677)	total: 17m 14s	remaining: 11m 29s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7382487702
bestIteration = 1677

Shrink model to first 1678 iterations.
Fold AUC: 0.73825

===== FOLD 1 =====
0:	test: 0.6876390	best: 0.6876390 

In [17]:
new_features = [
    # worst loan
    "worst_loan_paym_mean",
    "worst_loan_util",
    "worst_loan_overdue",
    "worst_loan_paym_to_mean",
    "worst_loan_util_to_mean_util",

    # last loan
    "last_loan_paym_mean",
    "last_loan_util",
    "last_loan_outstanding",
    "last_vs_mean_paym",
    "last_vs_mean_util",
    "last_vs_worst_paym",

    # dispersion
    "loan_paym_std",
    "loan_util_std",
    "loan_overdue_std",
    "loan_paym_max_minus_mean",
    "loan_util_max_minus_mean",
]

In [18]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance()
})

# только новые фичи
new_feat_importance = feature_importance[
    feature_importance["feature"].isin(new_features)
].sort_values("importance", ascending=False)

print(new_feat_importance)

                          feature  importance
114                last_loan_util    2.108120
131            last_vs_worst_paym    1.950424
116         last_loan_outstanding    1.590421
127      loan_util_max_minus_mean    1.478892
129             last_vs_mean_util    1.231421
130             last_vs_mean_paym    1.104118
126      loan_paym_max_minus_mean    0.960508
117                 loan_paym_std    0.819798
112           last_loan_paym_mean    0.807143
132  worst_loan_util_to_mean_util    0.718279
120                 loan_util_std    0.533657
107               worst_loan_util    0.520324
123              loan_overdue_std    0.419157
105          worst_loan_paym_mean    0.277581
108            worst_loan_overdue    0.069445


In [19]:
total_importance = feature_importance["importance"].sum()
new_importance = new_feat_importance["importance"].sum()

print("New features total importance:", new_importance)
print("Share of total importance:", new_importance / total_importance)

New features total importance: 14.589289065109323
Share of total importance: 0.14589289065109315


In [20]:
for col in new_features:
    if col in X.columns:
        auc = roc_auc_score(y, X[col])
        print(f"{col}: {auc:.5f}")

worst_loan_paym_mean: 0.53476
worst_loan_util: 0.48952
worst_loan_overdue: 0.49135
worst_loan_util_to_mean_util: 0.50965
last_loan_paym_mean: 0.58107
last_loan_util: 0.49424
last_loan_outstanding: 0.51710
last_vs_mean_paym: 0.47835
last_vs_mean_util: 0.54353
last_vs_worst_paym: 0.56037
loan_paym_std: 0.42435
loan_util_std: 0.45043
loan_overdue_std: 0.54414
loan_paym_max_minus_mean: 0.41120
loan_util_max_minus_mean: 0.45097


## modeling 14.04

In [17]:
cat_params = {
    'iterations': 3000,
    'depth': 8,
    'learning_rate': 0.03,
    'l2_leaf_reg': 7,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'class_weights': class_weights_dict,
    'early_stopping_rounds': 200,
    'verbose': 200
}

In [18]:
X = data.drop(['flag', 'id'], axis=1)
y = data["flag"]

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== FOLD {fold} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)

    model.fit(X_train, y_train, eval_set=(X_val, y_val))

    preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = preds

    fold_auc = roc_auc_score(y_val, preds)
    scores.append(fold_auc)

    print(f"Fold AUC: {fold_auc:.5f}")

print("\nMean AUC:", np.mean(scores))
print("OOF AUC:", roc_auc_score(y, oof_preds))


===== FOLD 0 =====
0:	test: 0.6827996	best: 0.6827996 (0)	total: 722ms	remaining: 36m 5s
200:	test: 0.7237999	best: 0.7237999 (200)	total: 1m 56s	remaining: 26m 59s
400:	test: 0.7304127	best: 0.7304127 (400)	total: 3m 53s	remaining: 25m 16s
600:	test: 0.7343932	best: 0.7343932 (600)	total: 5m 50s	remaining: 23m 19s
800:	test: 0.7364060	best: 0.7364060 (800)	total: 7m 46s	remaining: 21m 21s
1000:	test: 0.7373331	best: 0.7373331 (1000)	total: 9m 42s	remaining: 19m 22s
1200:	test: 0.7378167	best: 0.7378167 (1200)	total: 11m 46s	remaining: 17m 37s
1400:	test: 0.7381992	best: 0.7381992 (1400)	total: 13m 44s	remaining: 15m 40s
1600:	test: 0.7382216	best: 0.7382384 (1594)	total: 15m 38s	remaining: 13m 39s
1800:	test: 0.7383253	best: 0.7383273 (1799)	total: 17m 33s	remaining: 11m 41s
2000:	test: 0.7382706	best: 0.7383475 (1841)	total: 19m 30s	remaining: 9m 44s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7383475277
bestIteration = 1841

Shrink model to first 1842 itera

In [24]:
new_features = [
    "last_vs_prev_util",
    "last_vs_prev_paym",
    "util_rank",
    "paym_rank",
    "last_vs_last3_util",
    "last_vs_last3_paym",
    "last_vs_std_util",
    "last_vs_std_paym",
    "last_util_x_std",
    "last_paym_x_std",
    "last_vs_mean_util_x_std",
    "last_vs_mean_paym_x_std",
]

In [27]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance()
})

# только новые фичи
new_feat_importance = feature_importance[
    feature_importance["feature"].isin(new_features)
].sort_values("importance", ascending=False)

print(new_feat_importance)

                     feature  importance
152         last_vs_std_paym    0.831587
151         last_vs_std_util    0.694093
155  last_vs_mean_util_x_std    0.605057
156  last_vs_mean_paym_x_std    0.483869
154          last_paym_x_std    0.380049
153          last_util_x_std    0.254936


In [28]:
total_importance = feature_importance["importance"].sum()
new_importance = new_feat_importance["importance"].sum()

print("New features total importance:", new_importance)
print("Share of total importance:", new_importance / total_importance)

New features total importance: 3.249591577906246
Share of total importance: 0.03249591577906244


In [29]:
for col in new_features:
    if col in X.columns:
        auc = roc_auc_score(y, X[col])
        print(f"{col}: {auc:.5f}")

last_vs_std_util: 0.49264
last_vs_std_paym: 0.49813
last_util_x_std: 0.45512
last_paym_x_std: 0.47488
last_vs_mean_util_x_std: 0.54280
last_vs_mean_paym_x_std: 0.47256


# 6. Final Model

This section evaluates an alternative approach provided externally.

Important:
The experiment is included for comparison purposes only.
It demonstrates that performance improvements can be achieved by altering data assumptions.

**Final ensemble AUC: ~0.742**

## списки фич, параметры моделей

In [4]:
xgb_features = [
    # --- core payment behavior ---
    "enc_paym_0_mean", "enc_paym_1_mean", "enc_paym_2_mean", "enc_paym_3_mean",
    "enc_paym_4_mean", "enc_paym_5_mean", "enc_paym_6_mean", "enc_paym_7_mean",
    "enc_paym_8_mean", "enc_paym_9_mean", "enc_paym_10_mean", "enc_paym_11_mean",
    "enc_paym_12_mean", "enc_paym_13_mean", "enc_paym_14_mean", "enc_paym_15_mean",
    "enc_paym_16_mean", "enc_paym_17_mean", "enc_paym_18_mean", "enc_paym_19_mean",
    "enc_paym_20_mean", "enc_paym_21_mean", "enc_paym_22_mean", "enc_paym_23_mean",

    # --- sums ---
    "enc_paym_0_sum", "enc_paym_1_sum", "enc_paym_2_sum", "enc_paym_3_sum",
    "enc_paym_4_sum", "enc_paym_5_sum", "enc_paym_6_sum", "enc_paym_7_sum",
    "enc_paym_8_sum", "enc_paym_9_sum", "enc_paym_10_sum", "enc_paym_11_sum",
    "enc_paym_19_sum", "enc_paym_20_sum", "enc_paym_21_sum", "enc_paym_22_sum",
    "enc_paym_23_sum",

    # --- max ---
    "enc_paym_0_max", "enc_paym_1_max",

    # --- behavioral aggregates ---
    "client_mean_bad_months_per_product",
    "client_mean_nonzero_paym_share",
    "client_mean_weighted_paym_score",
    "client_mean_paym_status",
    "client_mean_severe_bad_months",
    "client_max_weighted_paym_score",
    "client_worst_loan_max",

    # --- advanced behavior ---
    "recent_bad_ratio_max",
    "decayed_paym_score_mean",
    "decayed_paym_score_max",
    "worst_paym_ever",

    # --- dynamics ---
    "paym_std",
    "paym_switches",
    "paym_trend_lr",
    "paym_trend_abs",

    # --- relative ---
    "recent_vs_old",
    "recent_vs_old_early",
    "last_vs_mean",

    # --- NEW (самое важное) ---
    "last_vs_prev_util_last",
    "last_vs_prev_paym_last",
    "last_vs_last3_util_last",
    "last_vs_last3_paym_last",

    # --- ranks ---
    "util_rank_last",
    "paym_rank_last",

    # --- normalized ---
    "last_vs_std_util",
    "last_vs_std_paym",

    # --- interactions ---
    "last_util_x_std",
    "last_paym_x_std",
    "last_vs_mean_util_x_std",
    "last_vs_mean_paym_x_std",
]

lgb_features = [
    # --- ratios ---
    "bad_to_total_ratio",
    "util_per_loan",
    "util_stability",

    # --- credit structure ---
    "pre_loans_credit_cost_rate_mean",
    "pre_loans_outstanding_mean",
    "pre_loans_outstanding_max",
    "pre_loans_credit_limit_mean",
    "pre_loans_credit_limit_max",
    "pre_loans_next_pay_summ_mean",
    "pre_loans_next_pay_summ_max",
    "pre_loans_max_overdue_sum_max",

    # --- time ---
    "pre_since_opened_min",
    "pre_since_opened_max",
    "pre_since_opened_mean",
    "pre_since_confirmed_min",
    "pre_since_confirmed_max",
    "pre_since_confirmed_mean",

    "pre_pterm_min",
    "pre_pterm_max",
    "pre_pterm_mean",
    "pre_fterm_min",
    "pre_fterm_max",
    "pre_fterm_mean",

    "pre_till_pclose_min",
    "pre_till_pclose_max",
    "pre_till_fclose_min",
    "pre_till_fclose_max",

    # --- utilization ---
    "pre_util_mean",
    "pre_util_max",
    "mean_utilization_ratio_max",
    "max_utilization_ratio_max",
    "client_util_std_max",

    # --- flags ---
    "pclose_flag_mean",
    "fclose_flag_mean",

    # --- categorical ---
    "enc_loans_credit_type_nunique",
    "enc_loans_account_holder_type_nunique",
    "enc_loans_credit_status_nunique",
    "enc_loans_account_cur_nunique",

    # --- counts ---
    "pre_loans5_sum",
    "pre_loans530_sum",
    "client_products_count",
    "client_products_count_max",

    # --- zero ---
    "is_zero_util_mean",
    "is_zero_loans5_mean",
    "is_zero_loans5_sum",
    "is_zero_loans530_mean",
    "is_zero_loans530_sum",
    "is_zero_loans3060_mean",
    "is_zero_loans6090_mean",
    "is_zero_loans90_mean",
    "is_zero_over2limit_mean",
    "is_zero_maxover2limit_mean",

    # --- limits ---
    "pre_over2limit_max",

    # --- client ---
    "max_credit_share_max",
    "max_credit_share",

    # --- last/worst loans (как объекты) ---
    "worst_loan_paym_mean",
    "worst_loan_paym_max",
    "worst_loan_util",
    "worst_loan_overdue",
    "worst_loan_outstanding",
    "worst_loan_credit_limit",
    "worst_loan_overdue_to_limit",

    "last_loan_paym_mean",
    "last_loan_paym_max",
    "last_loan_util",
    "last_loan_overdue",
    "last_loan_outstanding",

    # --- dispersion ---
    "loan_paym_std",
    "loan_util_std",
    "loan_overdue_std",
    "loan_paym_max",
    "loan_util_max",
    "loan_util_mean",
    "loan_overdue_max",
    "loan_overdue_mean",

    "loan_paym_max_minus_mean",
    "loan_util_max_minus_mean",
    "loan_overdue_max_minus_mean",

    # --- relations ---
    "last_vs_mean_util",
    "last_vs_mean_paym",
    "last_vs_worst_paym",
    "worst_loan_util_to_mean_util",

    # --- misc ---
    "last_paym_status_max",
]

In [7]:
cat_params = {
    'iterations': 3000,
    'depth': 8,
    'learning_rate': 0.03,
    'l2_leaf_reg': 7,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'class_weights': class_weights_dict,
    'early_stopping_rounds': 200,
    'verbose': 200
}

lgb_params = {
    'n_estimators': 5000,
    'num_leaves': 31,
    'colsample_bytree': 0.7,
    'min_child_samples': 50,
    'subsample': 0.7,
    'learning_rate': 0.03,
    'max_depth': -1,
    'random_state': 42,
    'class_weight': class_weights_dict,
    'verbosity': -1
}

pos_weight = class_weights_dict[1] / class_weights_dict[0]
xgb_params = {
    'n_estimators': 5000,
    'learning_rate': 0.03,
    'max_depth': 6,
    'reg_lambda': 10,
    'reg_alpha': 0,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'auc',
    'random_state': 42,
    'scale_pos_weight': pos_weight,
    'early_stopping_rounds': 200,
    'verbosity': 1
}

## modeling

In [8]:
X = data.drop(['flag', 'id'], axis=1)
X_lgb = data[lgb_features]
X_xgb = data[xgb_features]

y = data["flag"]

X_np = X.to_numpy(dtype=np.float32)
X_lgb_np = X_lgb.to_numpy(dtype=np.float32)
X_xgb_np = X_xgb.to_numpy(dtype=np.float32)
y_np = y.to_numpy(dtype=np.int8)

print(set(lgb_features) & set(xgb_features))

print(X_lgb.shape)
print(X_xgb.shape)

set()
(3000000, 83)
(3000000, 73)


In [9]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

cat_models_auc = []
lgb_models_auc = []
xgb_models_auc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n========== FOLD {fold} ==========")

    X_train, X_val = X_np[train_idx], X_np[val_idx]
    X_train_lgb, X_val_lgb = X_lgb_np[train_idx], X_lgb_np[val_idx]
    X_train_xgb, X_val_xgb = X_xgb_np[train_idx], X_xgb_np[val_idx]
    y_train, y_val = y_np[train_idx], y_np[val_idx]

    # ===== Обработка inf =====
    for arr_train, arr_val in [(X_train, X_val), (X_train_lgb, X_val_lgb), (X_train_xgb, X_val_xgb)]:
        for i in range(arr_train.shape[1]):
            col_train = arr_train[:, i]
            col_val = arr_val[:, i]

            median_val = np.nanmedian(col_train[np.isfinite(col_train)])

            col_train[~np.isfinite(col_train)] = median_val
            col_val[~np.isfinite(col_val)] = median_val

    # ===== CatBoost =====
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True,
        verbose=200
    )

    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    auc_cat = roc_auc_score(y_val, oof_cat[val_idx])
    best_iter_cat = cat_model.get_best_iteration()

    print(f"[CatBoost]  AUC: {auc_cat:.6f} | best_iter: {best_iter_cat}")

    cat_models_auc.append((auc_cat, cat_model))

    # ===== LightGBM =====
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_train_lgb, y_train,
        eval_set=[(X_val_lgb, y_val)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(200)
        ]
    )

    oof_lgb[val_idx] = lgb_model.predict_proba(X_val_lgb)[:, 1]
    auc_lgb = roc_auc_score(y_val, oof_lgb[val_idx])
    best_iter_lgb = lgb_model.best_iteration_

    print(f"[LightGBM] AUC: {auc_lgb:.6f} | best_iter: {best_iter_lgb}")

    lgb_models_auc.append((auc_lgb, lgb_model))

    # ===== XGBoost =====
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train_xgb, y_train,
        eval_set=[(X_val_xgb, y_val)],
        verbose=200
    )

    oof_xgb[val_idx] = xgb_model.predict_proba(X_val_xgb)[:, 1]
    auc_xgb = roc_auc_score(y_val, oof_xgb[val_idx])
    best_iter_xgb = xgb_model.best_iteration

    print(f"[XGBoost]  AUC: {auc_xgb:.6f} | best_iter: {best_iter_xgb}")

    xgb_models_auc.append((auc_xgb, xgb_model))


print("\n========== OOF RESULTS ==========")
print("CatBoost AUC:", roc_auc_score(y, oof_cat))
print("LightGBM AUC:", roc_auc_score(y, oof_lgb))
print("XGBoost AUC:", roc_auc_score(y, oof_xgb))


# ===== STACKING =====
stack_X = np.vstack([oof_cat, oof_lgb, oof_xgb]).T
stack_oof = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(stack_X, y)):
    print(f"\n===== META FOLD {fold} =====")

    X_train_meta, X_val_meta = stack_X[train_idx], stack_X[val_idx]
    y_train_meta, y_val_meta = y.iloc[train_idx], y.iloc[val_idx]

    meta_model = CatBoostClassifier(
        iterations=1000,
        depth=6,
        l2_leaf_reg=5,
        learning_rate=0.03,
        loss_function='Logloss',
        eval_metric='AUC',
        iterations=2000,
        verbose=200
    )

    meta_model.fit(
        X_train_meta, y_train_meta,
        eval_set=(X_val_meta, y_val_meta),
        use_best_model=True
    )

    stack_oof[val_idx] = meta_model.predict_proba(X_val_meta)[:, 1]
    auc_meta = roc_auc_score(y_val_meta, stack_oof[val_idx])

    print(f"[META] AUC: {auc_meta:.6f}")


print("\n========== FINAL ==========")
print("STACKING AUC:", roc_auc_score(y, stack_oof))


========== FOLD 0 ==========
0:	test: 0.6807794	best: 0.6807794 (0)	total: 954ms	remaining: 47m 40s
200:	test: 0.7260924	best: 0.7260924 (200)	total: 2m 32s	remaining: 35m 18s
400:	test: 0.7323657	best: 0.7323657 (400)	total: 4m 53s	remaining: 31m 41s
600:	test: 0.7362737	best: 0.7362737 (600)	total: 7m 4s	remaining: 28m 15s
800:	test: 0.7383358	best: 0.7383358 (800)	total: 9m 27s	remaining: 25m 57s
1000:	test: 0.7394563	best: 0.7394586 (999)	total: 11m 56s	remaining: 23m 51s
1200:	test: 0.7401155	best: 0.7401364 (1196)	total: 14m 19s	remaining: 21m 27s
1400:	test: 0.7403993	best: 0.7403993 (1400)	total: 16m 46s	remaining: 19m 8s
1600:	test: 0.7405992	best: 0.7406630 (1577)	total: 19m 11s	remaining: 16m 45s
1800:	test: 0.7406357	best: 0.7406761 (1654)	total: 21m 38s	remaining: 14m 24s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7406760928
bestIteration = 1654

Shrink model to first 1655 iterations.
[CatBoost]  AUC: 0.740676 | best_iter: 1654
Training until val

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] AUC: 0.738766 | best_iter: 1468
[0]	validation_0-auc:0.65967
[200]	validation_0-auc:0.69535
[400]	validation_0-auc:0.69893
[600]	validation_0-auc:0.70018
[800]	validation_0-auc:0.70083
[1000]	validation_0-auc:0.70111
[1200]	validation_0-auc:0.70104
[1322]	validation_0-auc:0.70111
[XGBoost]  AUC: 0.701186 | best_iter: 1122

========== FOLD 1 ==========
0:	test: 0.6778211	best: 0.6778211 (0)	total: 696ms	remaining: 34m 45s
200:	test: 0.7239287	best: 0.7239287 (200)	total: 2m 22s	remaining: 33m 8s
400:	test: 0.7304349	best: 0.7304349 (400)	total: 4m 44s	remaining: 30m 42s
600:	test: 0.7344030	best: 0.7344030 (600)	total: 7m 8s	remaining: 28m 30s
800:	test: 0.7363293	best: 0.7363293 (800)	total: 9m 27s	remaining: 25m 58s
1000:	test: 0.7375687	best: 0.7375687 (1000)	total: 11m 43s	remaining: 23m 25s
1200:	test: 0.7383153	best: 0.7383153 (1200)	total: 13m 57s	remaining: 20m 54s
1400:	test: 0.7387523	best: 0.7387523 (1400)	total: 16m 20s	remaining: 18m 38s
1600:	test: 0.7388154	bes

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] AUC: 0.737041 | best_iter: 1571
[0]	validation_0-auc:0.65937
[200]	validation_0-auc:0.69538
[400]	validation_0-auc:0.69889
[600]	validation_0-auc:0.70018
[800]	validation_0-auc:0.70066
[1000]	validation_0-auc:0.70092
[1200]	validation_0-auc:0.70103
[1400]	validation_0-auc:0.70114
[1600]	validation_0-auc:0.70103
[1627]	validation_0-auc:0.70100
[XGBoost]  AUC: 0.701175 | best_iter: 1427

========== FOLD 2 ==========
0:	test: 0.6849727	best: 0.6849727 (0)	total: 656ms	remaining: 32m 46s
200:	test: 0.7309658	best: 0.7309658 (200)	total: 2m 21s	remaining: 32m 48s
400:	test: 0.7371710	best: 0.7371710 (400)	total: 4m 41s	remaining: 30m 27s
600:	test: 0.7407115	best: 0.7407115 (600)	total: 7m 7s	remaining: 28m 26s
800:	test: 0.7427149	best: 0.7427149 (800)	total: 9m 28s	remaining: 26m 2s
1000:	test: 0.7436643	best: 0.7436666 (999)	total: 11m 47s	remaining: 23m 32s
1200:	test: 0.7441927	best: 0.7441927 (1200)	total: 14m 3s	remaining: 21m 2s
1400:	test: 0.7443951	best: 0.7444217 (1320

...

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] AUC: 0.737656 | best_iter: 1750
[0]	validation_0-auc:0.65823
[200]	validation_0-auc:0.69489
[400]	validation_0-auc:0.69904
[600]	validation_0-auc:0.70022
[800]	validation_0-auc:0.70080
[1000]	validation_0-auc:0.70100
[1200]	validation_0-auc:0.70105
[1232]	validation_0-auc:0.70103
[XGBoost]  AUC: 0.701085 | best_iter: 1032

========== FOLD 4 ==========
0:	test: 0.6792036	best: 0.6792036 (0)	total: 728ms	remaining: 36m 23s
200:	test: 0.7261446	best: 0.7261446 (200)	total: 2m 33s	remaining: 35m 31s
400:	test: 0.7327444	best: 0.7327444 (400)	total: 5m 2s	remaining: 32m 39s
600:	test: 0.7364004	best: 0.7364004 (600)	total: 7m 32s	remaining: 30m 4s
800:	test: 0.7384076	best: 0.7384090 (799)	total: 10m 6s	remaining: 27m 45s
1000:	test: 0.7396052	best: 0.7396052 (1000)	total: 12m 30s	remaining: 24m 59s
1200:	test: 0.7403379	best: 0.7403474 (1199)	total: 14m 50s	remaining: 22m 13s
1400:	test: 0.7407401	best: 0.7407436 (1395)	total: 17m 10s	remaining: 19m 36s
1600:	test: 0.7410002	bes

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] AUC: 0.739163 | best_iter: 1709
[0]	validation_0-auc:0.66032
[200]	validation_0-auc:0.69634
[400]	validation_0-auc:0.70043
[600]	validation_0-auc:0.70176
[800]	validation_0-auc:0.70225
[1000]	validation_0-auc:0.70255
[1200]	validation_0-auc:0.70254
[1247]	validation_0-auc:0.70251
[XGBoost]  AUC: 0.702564 | best_iter: 1047

========== OOF RESULTS ==========
CatBoost AUC: 0.7412106295439781
LightGBM AUC: 0.7389532762022821
XGBoost AUC: 0.7021392043744689


SyntaxError: keyword argument repeated: iterations (394158217.py, line 107)

In [11]:
# ===== STACKING =====
stack_X = np.vstack([oof_cat, oof_lgb, oof_xgb]).T
stack_oof = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(stack_X, y)):
    print(f"\n===== META FOLD {fold} =====")

    X_train_meta, X_val_meta = stack_X[train_idx], stack_X[val_idx]
    y_train_meta, y_val_meta = y.iloc[train_idx], y.iloc[val_idx]

    meta_model = CatBoostClassifier(
        iterations=2000,
        depth=6,
        l2_leaf_reg=5,
        learning_rate=0.03,
        loss_function='Logloss',
        eval_metric='AUC',
        verbose=200
    )

    meta_model.fit(
        X_train_meta, y_train_meta,
        eval_set=(X_val_meta, y_val_meta),
        use_best_model=True
    )

    stack_oof[val_idx] = meta_model.predict_proba(X_val_meta)[:, 1]
    auc_meta = roc_auc_score(y_val_meta, stack_oof[val_idx])

    print(f"[META] AUC: {auc_meta:.6f}")

print("\n========== FINAL ==========")
print("STACKING AUC:", roc_auc_score(y, stack_oof))


===== META FOLD 0 =====
0:	test: 0.7226107	best: 0.7226107 (0)	total: 235ms	remaining: 7m 49s
200:	test: 0.7413465	best: 0.7413465 (200)	total: 46.6s	remaining: 6m 56s
400:	test: 0.7415334	best: 0.7415345 (399)	total: 1m 34s	remaining: 6m 15s
600:	test: 0.7415378	best: 0.7415421 (549)	total: 2m 20s	remaining: 5m 27s
800:	test: 0.7415390	best: 0.7415446 (755)	total: 3m 7s	remaining: 4m 40s
1000:	test: 0.7414953	best: 0.7415446 (755)	total: 3m 53s	remaining: 3m 53s
1200:	test: 0.7414522	best: 0.7415446 (755)	total: 4m 41s	remaining: 3m 7s
1400:	test: 0.7413729	best: 0.7415446 (755)	total: 5m 30s	remaining: 2m 21s
1600:	test: 0.7413042	best: 0.7415446 (755)	total: 6m 19s	remaining: 1m 34s
1800:	test: 0.7412376	best: 0.7415446 (755)	total: 7m 7s	remaining: 47.3s
1999:	test: 0.7411884	best: 0.7415446 (755)	total: 7m 55s	remaining: 0us

bestTest = 0.7415446216
bestIteration = 755

Shrink model to first 756 iterations.
[META] AUC: 0.741545

===== META FOLD 1 =====
0:	test: 0.7210305	best: 0.

In [15]:
models_dir = "models"
os.makedirs(models_dir, exist_ok=True)

cat_dir = os.path.join(models_dir, "catboost")
lgb_dir = os.path.join(models_dir, "lightgbm")
xgb_dir = os.path.join(models_dir, "xgboost")
meta_dir = os.path.join(models_dir, "meta")

for d in [cat_dir, lgb_dir, xgb_dir, meta_dir]:
    os.makedirs(d, exist_ok=True)

np.save(os.path.join(models_dir, "oof_cat.npy"), oof_cat)
np.save(os.path.join(models_dir, "oof_lgb.npy"), oof_lgb)
np.save(os.path.join(models_dir, "oof_xgb.npy"), oof_xgb)
np.save(os.path.join(models_dir, "oof_stack.npy"), stack_oof)

print("OOF saved")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

splits = []
for train_idx, val_idx in skf.split(np.zeros(len(y)), y):
    splits.append({
        "train_idx": train_idx.tolist(),
        "val_idx": val_idx.tolist()
    })

with open(os.path.join(models_dir, "splits.json"), "w") as f:
    json.dump(splits, f)

print("Splits saved")

cat_meta = []
lgb_meta = []
xgb_meta = []

for i, (auc, model) in enumerate(cat_models_auc):
    path = os.path.join(cat_dir, f"cat_fold_{i}.cbm")
    model.save_model(path)
    cat_meta.append({"fold": i, "auc": float(auc), "path": path})

for i, (auc, model) in enumerate(lgb_models_auc):
    path = os.path.join(lgb_dir, f"lgb_fold_{i}.pkl")
    joblib.dump(model, path)
    lgb_meta.append({"fold": i, "auc": float(auc), "path": path})

for i, (auc, model) in enumerate(xgb_models_auc):
    path = os.path.join(xgb_dir, f"xgb_fold_{i}.json")
    model.save_model(path)
    xgb_meta.append({"fold": i, "auc": float(auc), "path": path})

meta_model.save_model(os.path.join(meta_dir, "meta_model.cbm"))

print("Models saved")

summary = {
    "cat_auc": float(roc_auc_score(y, oof_cat)),
    "lgb_auc": float(roc_auc_score(y, oof_lgb)),
    "xgb_auc": float(roc_auc_score(y, oof_xgb)),
    "stack_auc": float(roc_auc_score(y, stack_oof)),
    "cat_models": cat_meta,
    "lgb_models": lgb_meta,
    "xgb_models": xgb_meta
}

with open(os.path.join(models_dir, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved")

OOF saved
Splits saved
Models saved
Summary saved


# 7. External Approach Comparison

This section evaluates an alternative approach provided externally.

Important:
The experiment is included for comparison purposes only.
It demonstrates that performance improvements can be achieved by altering data assumptions.

## experiment 20.04

In [3]:
data = pd.read_csv('full_data13.csv')

In [16]:
data_clean = data.drop(data[(data.is_zero_loans90_min == 0) & (data.flag == 0)].index)

In [17]:
mask = (data.is_zero_loans90_min == 0)

subset = data[mask]

print(subset['flag'].value_counts(normalize=True))

flag
0    0.941198
1    0.058802
Name: proportion, dtype: float64


In [18]:
mask_conflict = (data.is_zero_loans90_min == 0) & (data.flag == 0)

conflicts = data[mask_conflict]

print(len(conflicts), len(conflicts) / len(data))

367362 0.122454


In [30]:
def run_cv_catboost(data, target_col='flag', n_splits=3):

    X = data.drop(columns=[target_col])
    y = data[target_col]

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    oof_preds = np.zeros(len(data))
    fold_aucs = []

    classes = np.array([0, 1])
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    class_weights_dict = {0: weights[0], 1: weights[1]}

    cat_params = {
        'iterations': 1500,
        'depth': 6,
        'learning_rate': 0.03,
        'l2_leaf_reg': 7,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'class_weights': class_weights_dict,
        'early_stopping_rounds': 200,
        'verbose': 200,
        'random_seed': 42
    }

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostClassifier(**cat_params)

        model.fit(
            X_train, y_train,
            eval_set=(X_val, y_val),
            use_best_model=True
        )

        preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = preds

        auc = roc_auc_score(y_val, preds)
        fold_aucs.append(auc)

        print(f"Fold {fold}: AUC = {auc:.5f}")

    print("\nMean AUC:", np.mean(fold_aucs))
    print("Std AUC:", np.std(fold_aucs))

    return np.mean(fold_aucs)

In [31]:
print("=== ORIGINAL DATA ===")
auc_original = run_cv_catboost(data)

print("\n=== CLEAN DATA ===")
auc_clean = run_cv_catboost(data_clean)

print("\n=== COMPARISON ===")
print(f"Original AUC: {auc_original:.5f}")
print(f"Clean AUC:    {auc_clean:.5f}")
print(f"Diff:         {auc_clean - auc_original:.5f}")

=== ORIGINAL DATA ===
0:	test: 0.6728527	best: 0.6728527 (0)	total: 442ms	remaining: 11m 2s
200:	test: 0.7200663	best: 0.7200663 (200)	total: 1m 31s	remaining: 9m 48s
400:	test: 0.7278284	best: 0.7278284 (400)	total: 2m 59s	remaining: 8m 11s
600:	test: 0.7327422	best: 0.7327422 (600)	total: 4m 31s	remaining: 6m 46s
800:	test: 0.7356316	best: 0.7356316 (800)	total: 6m 5s	remaining: 5m 19s
1000:	test: 0.7374657	best: 0.7374657 (1000)	total: 7m 37s	remaining: 3m 48s
1200:	test: 0.7386278	best: 0.7386278 (1200)	total: 9m 8s	remaining: 2m 16s
1400:	test: 0.7394504	best: 0.7394504 (1400)	total: 10m 41s	remaining: 45.4s
1499:	test: 0.7396843	best: 0.7396843 (1499)	total: 11m 25s	remaining: 0us

bestTest = 0.7396842998
bestIteration = 1499

Fold 0: AUC = 0.73968
0:	test: 0.6782771	best: 0.6782771 (0)	total: 477ms	remaining: 11m 54s
200:	test: 0.7253948	best: 0.7253948 (200)	total: 1m 32s	remaining: 9m 57s
400:	test: 0.7330897	best: 0.7330897 (400)	total: 3m 5s	remaining: 8m 27s
600:	test: 0.73

In [28]:
print(
    ((data['is_zero_loans90_min'] == 0) & (data['flag'] == 0)).sum()
    / (data['is_zero_loans90_min'] == 0).sum()
)

0.9411984740451894


In [29]:
print("Shape:", data.shape)
print("Memory (MB):", data.memory_usage(deep=True).sum() / 1024**2)

print("\nColumns:", len(data.columns))
print("Target distribution:\n", data['flag'].value_counts(normalize=True))

print("Clean shape:", data_clean.shape)
print("Removed rows:", len(data) - len(data_clean))
print("Removed %:", 1 - len(data_clean) / len(data))

Shape: (3000000, 160)
Memory (MB): 901.222354888916

Columns: 160
Target distribution:
 flag
0    0.964519
1    0.035481
Name: proportion, dtype: float64
Clean shape: (2632638, 160)
Removed rows: 367362
Removed %: 0.12245399999999995


In [24]:
data.groupby('is_zero_loans90_min')['flag'].mean()

is_zero_loans90_min
0    0.058802
1    0.031993
Name: flag, dtype: float64